## 1. Objective

Build the first version of the **Feature Proposal Matrix** and the **Modeling Matrix**
for the project's supervised-learning stage: `X -> volume_demanda_proxy`.

**Target (already fixed, not redefined here)**: because TISS/ANS microdata has no
persistent beneficiary/patient identifier (`documents/extras/target_definition_potencial_pacientes.md`
established this), the operational target is `volume_demanda_proxy = SUM(QT_ITEM_EVENTO_INFORMADO)`
at `ID_PLANO x treatment_line_primary x UF_PRESTADOR x MES` granularity -- a **utilization
volume proxy**, not a literal patient count.

**Segmentation context**: `STATUS_ATUACAO` comes from the **operator x municipality**
segmentation method (notebooks 11/12: `activity_matrix_by_municipality.parquet` /
`active_operator_municipality_pairs.parquet`), not the operator-level segmentation
(notebook 10). Both exist in the repository; Section 4.2 shows they disagree for a
meaningful share of operators.

Notebook 12 applies that method once, to the full year. That is right for *describing*
2025, but wrong for *selecting* a forecasting population: an operator would enter the
modeling data because of claims it billed later in the year. This notebook therefore
re-applies the notebook 11/12 method **as of each month**, on the claims dated before that
month, and a row enters the modeling matrix only if its operator was ATUANTE at that point
and its combination had already been billed (Sections 4 and 6.1). The full-year
classification is kept only for traceability and for the notebook 10 comparison.

**Ground rules carried over from the target-definition work**: do not treat events,
procedures, or beneficiaries as patients; do not use future information; do not select
features by correlation alone; do not inherit K-Means' `_LOG`/`_SCALED` transformations
into this matrix -- business-value features are built from `df_norte` or the *raw*
(non-scaled) activity matrices, never the standardized ones.

**Population**: `df_norte` is the Medicina de Grupo subset of the treated base (`data_pipeline/08_filter_medicina_de_grupo.ipynb`), adopted as the project's official modeling population; notebooks 09-12 already run on it, so `STATUS_ATUACAO` below is computed within that population, not the full North region base.


## 2. Unit of Observation

`ID_PLANO x treatment_line_primary x UF_PRESTADOR x MES` -- fixed by the target
definition, not chosen here. Row count, duplicates, missing values, temporal coverage
and entity counts are verified below once `treatment_line_primary` is built (Section 3).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

_INPUT_PATH = Path("../data/processed/north_region_mama_2025_data_en_medicina_de_grupo.parquet")
if not _INPUT_PATH.is_file():
    raise FileNotFoundError(
        "Missing Medicina de Grupo base. Run data_pipeline/08_filter_medicina_de_grupo.ipynb "
        f"first. Expected path: {_INPUT_PATH}"
    )
df_norte = pd.read_parquet(_INPUT_PATH)
print("df_norte shape:", df_norte.shape)

df_norte shape: (70004, 119)


## 3. Target Construction

`volume_demanda_proxy = SUM(QT_ITEM_EVENTO_INFORMADO)` by `ID_PLANO x
treatment_line_primary x UF_PRESTADOR x MES`, using the `treatment_line_primary` logic
already decomposed in notebook 04 (`IND_LINE_STAGING/NEOADJUVANT/ADJUVANT/ADVANCED`).

**A real problem has to be reported before this can be trusted**: an event can carry
more than one `IND_LINE_*` flag at once. This is checked directly, not assumed away.

In [2]:
line_cols = ["IND_LINE_STAGING", "IND_LINE_NEOADJUVANT", "IND_LINE_ADJUVANT", "IND_LINE_ADVANCED"]
n_lines_active = df_norte[line_cols].sum(axis=1)

print("Events by number of simultaneously active IND_LINE_* flags:")
print(n_lines_active.value_counts().sort_index())
print(f"\nRows with more than one line active: {(n_lines_active > 1).sum()} "
      f"({(n_lines_active > 1).mean() * 100:.2f}%)")

Events by number of simultaneously active IND_LINE_* flags:
1      232
2    24238
3    43979
4     1555
Name: count, dtype: int64

Rows with more than one line active: 69772 (99.67%)


**Finding**: 99.67% of events carry more than one `IND_LINE_*` flag simultaneously.
This is not noise -- the TUSS/SBOC crosswalk (`data/reference/tuss_sboc_crosswalk_breast_cancer.csv`)
deliberately maps ambiguous, low-specificity procedure codes to multiple candidate
treatment lines at once (e.g. a generic staging exam usable across staging, neoadjuvant,
and advanced care). No documented, business-validated tie-break rule exists anywhere in
this project for collapsing that into a single `treatment_line_primary` per event.

**A priority rule is applied below**: `advanced > adjuvant > neoadjuvant > staging`
(clinical-severity ordering). It was introduced here as an unvalidated placeholder and is
now validated in `19_treatment_line_validation.ipynb` -- as a *series-partition
convention*, not as a clinical label. That notebook reproduces the counts printed below
exactly, shows that raw billed volume per operator x state x month is identical under any
tie-break, and measures the one place the rule does reach the target: through the
measurement-quality rule below, which excludes a whole group when any of its rows is a
package. What remains open is the partner's confirmation of the reporting convention for
co-flagged systemic agents -- see Section 4.3.11 of the documentation.

In [3]:
conditions = [
    df_norte["IND_LINE_ADVANCED"] == 1,
    df_norte["IND_LINE_ADJUVANT"] == 1,
    df_norte["IND_LINE_NEOADJUVANT"] == 1,
    df_norte["IND_LINE_STAGING"] == 1,
]
choices = ["advanced", "adjuvant", "neoadjuvant", "staging"]

# Validated as a series-partition convention in notebook 19; unchanged here.
df_norte["treatment_line_primary"] = np.select(conditions, choices, default="unclassified")

print(df_norte["treatment_line_primary"].value_counts())
assert (df_norte["treatment_line_primary"] == "unclassified").sum() == 0

treatment_line_primary
adjuvant       42390
neoadjuvant    17564
advanced       10050
Name: count, dtype: int64


In [4]:
UNIT_KEYS = ["ID_PLANO", "treatment_line_primary", "UF_PRESTADOR", "MES"]

unit_counts = df_norte.groupby(UNIT_KEYS, observed=True).size()
print("Distinct unit-of-observation groups:", len(unit_counts))
print("Distinct ID_PLANO:", df_norte["ID_PLANO"].nunique())
print("Distinct treatment_line_primary:", df_norte["treatment_line_primary"].nunique())
print("Distinct UF_PRESTADOR:", df_norte["UF_PRESTADOR"].nunique())
print("Distinct MES:", df_norte["MES"].nunique(), "range:", df_norte["MES"].min(), "-", df_norte["MES"].max())
print("Nulls in unit keys:", df_norte[UNIT_KEYS].isna().sum().to_dict())

Distinct unit-of-observation groups: 10302
Distinct ID_PLANO: 1052
Distinct treatment_line_primary: 3
Distinct UF_PRESTADOR: 7
Distinct MES: 12 range: 2025-01-01 00:00:00 - 2025-12-01 00:00:00
Nulls in unit keys: {'ID_PLANO': 0, 'treatment_line_primary': 0, 'UF_PRESTADOR': 0, 'MES': 0}


In [5]:
df_norte["_BUNDLED_ROW"] = df_norte["IND_PACOTE_1"].eq(1)
target = (
    df_norte.groupby(UNIT_KEYS, observed=True)
    .agg(
        volume_demanda_proxy=("QT_ITEM_EVENTO_INFORMADO", lambda s: s.sum(min_count=1)),
        N_SOURCE_ROWS=("QT_ITEM_EVENTO_INFORMADO", "size"),
        N_BUNDLED_ROWS=("_BUNDLED_ROW", "sum"),
    )
    .reset_index()
)
target["BUNDLING_RATE"] = target["N_BUNDLED_ROWS"] / target["N_SOURCE_ROWS"]
target["TARGET_MEASUREMENT_STATUS"] = np.select(
    [target["volume_demanda_proxy"].isna(), target["BUNDLING_RATE"].gt(0)],
    ["OBSERVED_ALL_BUNDLED_UNKNOWN", "OBSERVED_PARTIALLY_BUNDLED"],
    default="OBSERVED_MEASURED",
)
target["TARGET_USABLE_FOR_TRAINING"] = target["TARGET_MEASUREMENT_STATUS"].eq("OBSERVED_MEASURED")

assert target["volume_demanda_proxy"].isna().equals(target["BUNDLING_RATE"].eq(1))

print("target shape:", target.shape)
print("Duplicated keys:", target.duplicated(subset=UNIT_KEYS).sum())
print("Target measurement status:")
print(target["TARGET_MEASUREMENT_STATUS"].value_counts())
print("\nBundling-rate distribution among affected groups:")
print(target.loc[target["BUNDLING_RATE"].gt(0), "BUNDLING_RATE"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))
target["volume_demanda_proxy"].describe()

target shape: (10302, 10)
Duplicated keys: 0
Target measurement status:
TARGET_MEASUREMENT_STATUS
OBSERVED_MEASURED               10148
OBSERVED_PARTIALLY_BUNDLED        128
OBSERVED_ALL_BUNDLED_UNKNOWN       26
Name: count, dtype: int64

Bundling-rate distribution among affected groups:
count    154.000000
mean       0.358420
std        0.325008
min        0.021739
25%        0.112745
50%        0.240385
75%        0.500000
90%        1.000000
95%        1.000000
99%        1.000000
max        1.000000
Name: BUNDLING_RATE, dtype: float64


count    10276.000000
mean        80.075029
std        224.601232
min          1.000000
25%          1.000000
50%          4.000000
75%         90.000000
max       6974.000000
Name: volume_demanda_proxy, dtype: float64

**Target validation**: 10,302 groups, zero duplicated keys. Of these, 26 groups (0.25%)
contain only bundled rows (`IND_PACOTE_1=1`) and therefore retain a null target. A further
128 groups contain both bundled and individually reported rows; their numerical sum is
an observable lower bound rather than a complete demand measurement. Both conditions are
recorded in `TARGET_MEASUREMENT_STATUS` and excluded from the current training rows. No
unknown or incomplete target is converted to zero.

## 4. STATUS_ATUACAO Integration — as of each month

`STATUS_ATUACAO` is sourced from the **operator x municipality** segmentation method
(notebooks 11/12), not the operator-level one (notebook 10). This is a deliberate choice
for this version of the matrix, not a claim that it is the "correct" one -- Section 4.2
shows the two disagree for a meaningful share of operators.

**Two versions are computed, for different purposes.**

* The **full-year** status, rolled up from notebook 12's output below, describes 2025. It
  is kept for traceability and for the notebook 10 comparison in Section 4.2, and it
  **never selects rows**: an operator classified on full-year activity can be ATUANTE only
  because of claims billed after the month being modeled.
* The **as-of** status (Section 4.0) re-applies the notebook 11/12 method to the claims
  dated before each month *s*, and it alone decides whether rows of month *s* can enter the
  modeling matrix (Section 6.1).

**A join-safety problem has to be solved first**: `active_operator_municipality_pairs.parquet`
is keyed by `(ID_PLANO, CD_MUNICIPIO_BENEFICIARIO, CD_MUNICIPIO_PRESTADOR)` -- finer than
`modeling_matrix_v1`'s key, which has no municipality dimension at all. Merging it
directly by `ID_PLANO` would duplicate every `modeling_matrix_v1` row by however many
municipality pairs that operator has (a duplicate-key violation of the merge-safety rule
this notebook otherwise enforces throughout). The municipality-level result is rolled up
to one row per operator -- one row per operator and month, for the as-of version --
**before** any merge.

In [6]:
all_muni = pd.read_parquet("../data/interim/north_region_mama_2025_activity_matrix_by_municipality.parquet")
active_muni = pd.read_parquet("../data/processed/active_operator_municipality_pairs.parquet")

pairs_per_op = all_muni.groupby("ID_PLANO").size().rename("n_municipality_pairs_total")
active_per_op = active_muni.groupby("ID_PLANO").size().rename("n_municipality_pairs_active")

rollup = pd.concat([pairs_per_op, active_per_op], axis=1).fillna(0)
rollup["n_municipality_pairs_active"] = rollup["n_municipality_pairs_active"].astype(int)
rollup["share_active_municipality_pairs"] = (
    rollup["n_municipality_pairs_active"] / rollup["n_municipality_pairs_total"]
)
rollup["STATUS_ATUACAO"] = np.where(rollup["n_municipality_pairs_active"] > 0, "ATUANTE", "MARGINAL")
rollup = rollup.reset_index()

assert rollup["ID_PLANO"].duplicated().sum() == 0, "rollup must be exactly one row per operator"
print("rollup shape (1 row per operator):", rollup.shape)
rollup.head()

rollup shape (1 row per operator): (1052, 5)


,ID_PLANO,n_municipality_pairs_total,n_municipality_pairs_active,share_active_municipality_pairs,STATUS_ATUACAO
0,1019993,4,0,0.000000,MARGINAL
1,1169447,3,1,0.333333,ATUANTE
2,1169667,1,0,0.000000,MARGINAL
3,1169698,6,2,0.333333,ATUANTE
4,1169764,2,0,0.000000,MARGINAL


### 4.0 As-of status — the notebook 11/12 method on prior claims only

`activity_status()` below is notebook 11 (pair features, `log1p` on the same two columns
notebook 09 log-transforms, standardisation), notebook 12 (K-Means, active cluster =
the one with the highest mean `total_eventos`) and the roll-up above, applied to whatever
slice of claims it receives. Before the monthly version is trusted, it is run on all 2025
claims and asserted to reproduce notebook 12's classification operator by operator.

K is held at notebook 12's selected value, 2, rather than re-selected every month, so that
what changes from month to month is the information available, not the method. January
has no prior claims, so no January row can be eligible.

In [7]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

MUNI_KEYS = ["ID_PLANO", "CD_MUNICIPIO_BENEFICIARIO", "CD_MUNICIPIO_PRESTADOR"]
K_ACTIVITY = 2       # notebook 12's selected k, held fixed (see markdown above)
RANDOM_STATE = 42    # notebook 12's seed


def activity_status(claims):
    # notebook 11: operator x municipality pair features, log1p on the two skewed ones
    pairs = (claims.groupby(MUNI_KEYS, observed=True)
             .agg(total_eventos=("ID_EVENTO_ATENCAO_SAUDE", "nunique"),
                  meses_ativos=("MES", "nunique"),
                  procedimentos_distintos=("CD_PROCEDIMENTO", "nunique"))
             .reset_index())
    X = StandardScaler().fit_transform(np.column_stack([
        np.log1p(pairs["total_eventos"]), pairs["meses_ativos"],
        np.log1p(pairs["procedimentos_distintos"])]))
    # notebook 12: K-Means; the active cluster is the one with the highest mean total_eventos
    labels = KMeans(n_clusters=K_ACTIVITY, random_state=RANDOM_STATE, n_init=10).fit_predict(X)
    active_cluster = pairs.assign(cluster=labels).groupby("cluster")["total_eventos"].mean().idxmax()
    pairs["active"] = labels == active_cluster
    # roll-up (cell above): one row per operator
    out = (pairs.groupby("ID_PLANO", observed=True)
           .agg(n_municipality_pairs_total=("active", "size"),
                n_municipality_pairs_active=("active", "sum"))
           .reset_index())
    out["share_active_municipality_pairs"] = (
        out["n_municipality_pairs_active"] / out["n_municipality_pairs_total"])
    out["STATUS_ATUACAO"] = np.where(out["n_municipality_pairs_active"] > 0, "ATUANTE", "MARGINAL")
    return out


# Verification: on all 2025 claims the re-implementation must reproduce notebook 12.
check = activity_status(df_norte).merge(rollup, on="ID_PLANO", how="outer",
                                        suffixes=("", "_nb11"), indicator=True)
assert check["_merge"].eq("both").all(), "operator sets differ from the notebook 11/12 roll-up"
for col in ["STATUS_ATUACAO", "n_municipality_pairs_total", "n_municipality_pairs_active"]:
    assert (check[col] == check[f"{col}_nb11"]).all(), f"{col} differs from notebook 12"
print(f"Re-implementation reproduces notebook 12 for all {len(check):,} operators.")

FULL_MONTHS = pd.date_range(df_norte["MES"].min(), df_norte["MES"].max(), freq="MS")
status_asof = pd.concat(
    [activity_status(df_norte[df_norte["MES"] < s]).assign(MES=s) for s in FULL_MONTHS[1:]],
    ignore_index=True)
assert not status_asof.duplicated(["ID_PLANO", "MES"]).any(), "as-of status must be one row per operator-month"

print("\nATUANTE operators as of each month (claims dated before that month only):")
print(status_asof[status_asof["STATUS_ATUACAO"] == "ATUANTE"].groupby("MES").size().to_string())

Re-implementation reproduces notebook 11 for all 1,052 operators.



ATUANTE operators as of each month (claims dated before that month only):
MES
2025-02-01    243
2025-03-01    346
2025-04-01    376
2025-05-01    357
2025-06-01    393
2025-07-01    396
2025-08-01    417
2025-09-01    433
2025-10-01    413
2025-11-01    430
2025-12-01    438


In [8]:
# The full-year status is reported for traceability only -- it no longer selects rows.
target_full_status = target.merge(rollup[["ID_PLANO", "STATUS_ATUACAO"]], on="ID_PLANO", how="left")
assert len(target_full_status) == len(target), "merge must not duplicate target rows"

print("Target groups by FULL-YEAR STATUS_ATUACAO (not used for selection):")
print(target_full_status["STATUS_ATUACAO"].value_counts(dropna=False).to_string())
print("\nDistinct operators by full-year status:")
print(target_full_status.groupby("STATUS_ATUACAO", dropna=False)["ID_PLANO"].nunique().to_string())

# How the two selections differ at the operator level.
full_year_atuante = set(rollup.loc[rollup["STATUS_ATUACAO"] == "ATUANTE", "ID_PLANO"])
ever_asof_atuante = set(status_asof.loc[status_asof["STATUS_ATUACAO"] == "ATUANTE", "ID_PLANO"])
print(f"\nATUANTE in the full-year classification: {len(full_year_atuante):,}")
print(f"ATUANTE as of at least one month:         {len(ever_asof_atuante):,}")
print(f"  full-year ATUANTE but never ATUANTE as of any month: {len(full_year_atuante - ever_asof_atuante):,}")
print(f"  ATUANTE as of some month but MARGINAL over the full year: {len(ever_asof_atuante - full_year_atuante):,}")


Target groups by FULL-YEAR STATUS_ATUACAO (not used for selection):
STATUS_ATUACAO
ATUANTE     8086
MARGINAL    2216

Distinct operators by full-year status:
STATUS_ATUACAO
ATUANTE     428
MARGINAL    624

ATUANTE in the full-year classification: 428
ATUANTE as of at least one month:         507
  full-year ATUANTE but never ATUANTE as of any month: 4
  ATUANTE as of some month but MARGINAL over the full year: 83


### 4.1 ATUANTE population filter — applied as of each month

The supervised modeling population is still restricted to ATUANTE operators, but the
restriction is applied **row by row, as of the row's month**: a row for month *s* is
eligible only if its operator was ATUANTE on the claims billed before *s*. The filter runs
in Section 6.1, after the lag features have been computed on the complete panel, so
removing a row never breaks another row's history.

**Why the status has to be computed as of each month.** Merging a full-year status
onto every month would place an early-2025 row in the population because of activity
observed months later, and any model evaluated at a within-2025 cutoff would then be
evaluated on a population chosen with future information. Keeping `STATUS_ATUACAO`
out of `X` does not prevent that; only a status computed from prior information does.

The cost is a thinner population in the early months, because the status is computed on
little history — which is the information position of a forecaster early in the year. MARGINAL
operators and months remain in the source data and in the target table; they are only kept
out of the modeling rows.


### 4.2 Divergence with the operator-level segmentation (notebook 10)

In [9]:
old_active = pd.read_parquet("../data/processed/active_operators.parquet")[["ID_PLANO"]]
old_active["STATUS_ATUACAO_OP_LEVEL"] = "ATUANTE"

cmp = rollup.merge(old_active, on="ID_PLANO", how="left")
cmp["STATUS_ATUACAO_OP_LEVEL"] = cmp["STATUS_ATUACAO_OP_LEVEL"].fillna("MARGINAL")

crosstab = pd.crosstab(cmp["STATUS_ATUACAO"], cmp["STATUS_ATUACAO_OP_LEVEL"])
print(crosstab)
disagreement = (cmp["STATUS_ATUACAO"] != cmp["STATUS_ATUACAO_OP_LEVEL"]).sum()
print(f"\nOperators where the two segmentations disagree: {disagreement} ({disagreement / len(cmp) * 100:.1f}%)")

STATUS_ATUACAO_OP_LEVEL  ATUANTE  MARGINAL
STATUS_ATUACAO                            
ATUANTE                      302       126
MARGINAL                      37       587

Operators where the two segmentations disagree: 163 (15.5%)


**The two definitions remain methodologically distinct.** They were built on different
feature sets (operator-level activity vs. operator x municipality activity) and
different K-Means runs -- disagreeing for a meaningful share of operators is expected,
not a bug. `STATUS_ATUACAO` is therefore definition-dependent rather than a stable observed
attribute. This version uses the municipality method consistently, and applies it as of each month
for row selection (Section 4.1), which is the temporally valid definition production use
requires.

## 5. Feature Inventory and Feature Proposal Matrix

Every candidate is checked against the fundamental rule before being classified: does it
have a business rationale, does it exist before the prediction moment, is it available at
the right granularity, is it not a direct contemporaneous derivation of the target, does
it avoid future information, is it not just an identifier, and is it not a redundant
transformation of another variable. Features are **not** selected by correlation with the
target -- none was computed for this purpose.

`_LOG` / `_SCALED` columns from the K-Means matrices (notebooks 09/11) are **not**
reused here. Notebook 14 additionally establishes that the non-hospital market-share
source can support network-size and concentration proposals, but not ownership labels.
Those fields remain pending until identifier linkage, join cardinality, reference dates,
prediction-time availability and national-versus-North geographic scope are verified;
proposal status does not place them in `X`.

In [10]:
feature_proposal_matrix = pd.DataFrame([
    # --- A. OPERATOR / PLAN ---
    dict(FEATURE_NAME="PORTE", SOURCE_DATASET="df_norte", SOURCE_COLUMN="PORTE",
         FEATURE_GROUP="A_OPERATOR_PLAN", DESCRIPTION="Operator size tier by total active beneficiaries",
         ORIGINAL_GRANULARITY="operator", MODEL_GRANULARITY="operator (static)", RAW_OR_DERIVED="RAW",
         TRANSFORMATION="none (ordinal already encoded as PORTE_ORDINAL)", AGGREGATION="none",
         EXPECTED_RELATIONSHIP="+", BUSINESS_RATIONALE="Larger operators tend to larger absolute volume",
         TEMPORAL_AVAILABILITY="pre-period (confirmed time-invariant per operator)", LEAKAGE_RISK="LOW",
         MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW (4 levels)", REDUNDANCY_RISK="low (not redundant with STATUS_ATUACAO, checked)",
         PRIORITY="HIGH", STATUS="APPROVED"),
    dict(FEATURE_NAME="PORTE_ORDINAL", SOURCE_DATASET="df_norte", SOURCE_COLUMN="PORTE_ORDINAL",
         FEATURE_GROUP="A_OPERATOR_PLAN", DESCRIPTION="Ordinal-encoded version of PORTE",
         ORIGINAL_GRANULARITY="operator", MODEL_GRANULARITY="operator (static)", RAW_OR_DERIVED="RAW",
         TRANSFORMATION="none", AGGREGATION="none", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Same as PORTE", TEMPORAL_AVAILABILITY="pre-period", LEAKAGE_RISK="LOW",
         MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="high with PORTE (same info, different format)",
         PRIORITY="HIGH", STATUS="APPROVED (use one of the two, not both)"),
    dict(FEATURE_NAME="NM_MODALIDADE_*", SOURCE_DATASET="df_norte", SOURCE_COLUMN="NM_MODALIDADE_* (6 one-hot cols)",
         FEATURE_GROUP="A_OPERATOR_PLAN", DESCRIPTION="Operator's institutional modality",
         ORIGINAL_GRANULARITY="operator", MODEL_GRANULARITY="operator (static)", RAW_OR_DERIVED="RAW (already one-hot)",
         TRANSFORMATION="none", AGGREGATION="none", EXPECTED_RELATIONSHIP="?",
         BUSINESS_RATIONALE="Modality may correlate with network strategy", TEMPORAL_AVAILABILITY="pre-period",
         LEAKAGE_RISK="LOW", MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="low",
         PRIORITY="MEDIUM", STATUS="CANDIDATE"),
    dict(FEATURE_NAME="ID_PLANO", SOURCE_DATASET="df_norte", SOURCE_COLUMN="ID_PLANO",
         FEATURE_GROUP="A_OPERATOR_PLAN", DESCRIPTION="Operator identifier",
         ORIGINAL_GRANULARITY="operator", MODEL_GRANULARITY="operator (key)", RAW_OR_DERIVED="RAW",
         TRANSFORMATION="NONE -- not numeric", AGGREGATION="none", EXPECTED_RELATIONSHIP="n/a",
         BUSINESS_RATIONALE="It is the key, not a feature", TEMPORAL_AVAILABILITY="n/a", LEAKAGE_RISK="n/a",
         MISSINGNESS_RISK="0%", CARDINALITY_RISK="HIGH (3289)", REDUNDANCY_RISK="n/a",
         PRIORITY="n/a", STATUS="EXCLUDE (as numeric feature; use as key/group only)"),

    # --- B. DEMOGRAPHIC ---
    dict(FEATURE_NAME="PCT_FEMININO", SOURCE_DATASET="df_norte", SOURCE_COLUMN="SEXO_Feminino",
         FEATURE_GROUP="B_DEMOGRAPHIC", DESCRIPTION="Share of group's rows that are female",
         ORIGINAL_GRANULARITY="event", MODEL_GRANULARITY="group (contemporaneous)", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="proportion", AGGREGATION="mean", EXPECTED_RELATIONSHIP="?",
         BUSINESS_RATIONALE="Sex composition may vary by treatment line", TEMPORAL_AVAILABILITY="contemporaneous to target",
         LEAKAGE_RISK="HIGH", MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="low",
         PRIORITY="LOW now / MEDIUM if lagged", STATUS="CANDIDATE (only as a trailing/lagged version)"),
    dict(FEATURE_NAME="PCT_AGE_60PLUS", SOURCE_DATASET="df_norte", SOURCE_COLUMN="FAIXA_ETARIA_ORDINAL",
         FEATURE_GROUP="B_DEMOGRAPHIC", DESCRIPTION="Share of group's rows in age >= 60",
         ORIGINAL_GRANULARITY="event", MODEL_GRANULARITY="group (contemporaneous)", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="threshold + proportion", AGGREGATION="mean", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Breast cancer incidence concentrates in older age groups", TEMPORAL_AVAILABILITY="contemporaneous",
         LEAKAGE_RISK="HIGH", MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="low",
         PRIORITY="LOW now / MEDIUM if lagged", STATUS="CANDIDATE (only as a trailing/lagged version)"),

    # --- C. GEOGRAPHIC ---
    dict(FEATURE_NAME="UF_PRESTADOR", SOURCE_DATASET="df_norte", SOURCE_COLUMN="UF_PRESTADOR",
         FEATURE_GROUP="C_GEOGRAPHIC", DESCRIPTION="State where the service was provided",
         ORIGINAL_GRANULARITY="event", MODEL_GRANULARITY="already a modeling key", RAW_OR_DERIVED="RAW",
         TRANSFORMATION="none", AGGREGATION="none", EXPECTED_RELATIONSHIP="n/a",
         BUSINESS_RATIONALE="Already the model's geographic key", TEMPORAL_AVAILABILITY="pre-determined",
         LEAKAGE_RISK="LOW", MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW (7)", REDUNDANCY_RISK="n/a (is a key)",
         PRIORITY="n/a", STATUS="KEY, not a feature to select"),
    dict(FEATURE_NAME="N_MUNICIPIOS_BENEFICIARIO_TRAILING", SOURCE_DATASET="df_norte (re-engineered)", SOURCE_COLUMN="CD_MUNICIPIO_BENEFICIARIO",
         FEATURE_GROUP="C_GEOGRAPHIC", DESCRIPTION="Distinct beneficiary-residence municipalities, trailing window",
         ORIGINAL_GRANULARITY="operator-month (trailing)", MODEL_GRANULARITY="operator (trailing)", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="nunique over a rolling window, NOT the nb09 annual aggregate", AGGREGATION="nunique",
         EXPECTED_RELATIONSHIP="+", BUSINESS_RATIONALE="Broader demand-side geographic reach",
         TEMPORAL_AVAILABILITY="strictly historical window ending before prediction month; not yet engineered", LEAKAGE_RISK="MEDIUM (if trailing) / HIGH (if reusing nb09/11 as-is)",
         MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="low",
         PRIORITY="MEDIUM", STATUS="CANDIDATE -- requires new engineering, not direct reuse"),
    dict(FEATURE_NAME="N_MUNICIPIOS_PRESTADOR_TRAILING", SOURCE_DATASET="df_norte (re-engineered)", SOURCE_COLUMN="CD_MUNICIPIO_PRESTADOR",
         FEATURE_GROUP="C_GEOGRAPHIC", DESCRIPTION="Distinct provider municipalities, trailing window",
         ORIGINAL_GRANULARITY="operator-month (trailing)", MODEL_GRANULARITY="operator (trailing)", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="nunique over a rolling window", AGGREGATION="nunique", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Broader provider-side geographic reach", TEMPORAL_AVAILABILITY="strictly historical window ending before prediction month; not yet engineered",
         LEAKAGE_RISK="MEDIUM (if trailing) / HIGH (as-is)", MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="low",
         PRIORITY="MEDIUM", STATUS="CANDIDATE -- requires historical-window engineering"),
    # --- D. NETWORK / CONCENTRATION (AUDITED IN NOTEBOOK 14) ---
    dict(FEATURE_NAME="N_PRESTADORES_NO_GRUPO", SOURCE_DATASET="Non-Hospital Market Share (notebook 14)", SOURCE_COLUMN="N_PRESTADORES_NO_GRUPO",
         FEATURE_GROUP="D_NETWORK", DESCRIPTION="Declared provider count repeated within a plan; plan-network size, not corporate-group ownership",
         ORIGINAL_GRANULARITY="operator-plan-provider row; constant within plan", MODEL_GRANULARITY="plan snapshot; no valid scalar operator roll-up",
         RAW_OR_DERIVED="RAW", TRANSFORMATION="first after asserting one value per plan",
         AGGREGATION="retain by plan; count-incomplete plans remain diagnostic and are excluded from feature roll-ups",
         EXPECTED_RELATIONSHIP="?", BUSINESS_RATIONALE="Plan-level network size may explain access capacity",
         TEMPORAL_AVAILABILITY="unknown source reference/publication date; must predate prediction cutoff", LEAKAGE_RISK="HIGH until dated",
         MISSINGNESS_RISK="1 of 46 audited plans is count-incomplete", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="same as observed count in 45 of 46 plans",
         PRIORITY="MEDIUM", STATUS="PENDING -- excluded from X until linkage/date/scope/cardinality gates pass"),
    dict(FEATURE_NAME="N_PRESTADORES", SOURCE_DATASET="Non-Hospital Market Share (notebook 14)", SOURCE_COLUMN="CD_CNES",
         FEATURE_GROUP="D_NETWORK", DESCRIPTION="Observed distinct providers in each plan network",
         ORIGINAL_GRANULARITY="operator-plan-provider row", MODEL_GRANULARITY="plan snapshot; operator roll-up requires a documented rule",
         RAW_OR_DERIVED="DERIVED", TRANSFORMATION="nunique(CD_CNES)",
         AGGREGATION="by operator and plan; incomplete plans retained for diagnostics but excluded from feature roll-ups",
         EXPECTED_RELATIONSHIP="+", BUSINESS_RATIONALE="Observed provider-network size",
         TEMPORAL_AVAILABILITY="unknown source reference/publication date; must predate prediction cutoff", LEAKAGE_RISK="HIGH until dated",
         MISSINGNESS_RISK="under-counted for count-incomplete plans", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="Spearman rho=-0.947 with HHI and -0.899 with MAX in count-complete plans",
         PRIORITY="MEDIUM", STATUS="PENDING -- excluded from X until linkage/date/scope/cardinality gates pass"),
    dict(FEATURE_NAME="MARKET_SHARE_TOTAL", SOURCE_DATASET="Non-Hospital Market Share (notebook 14)", SOURCE_COLUMN="MARKET_SHARE_PCT",
         FEATURE_GROUP="D_NETWORK", DESCRIPTION="Weight-conservation diagnostic after within-plan normalization",
         ORIGINAL_GRANULARITY="operator-plan-provider row", MODEL_GRANULARITY="plan diagnostic", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="p_j = MARKET_SHARE_PCT / sum(MARKET_SHARE_PCT)", AGGREGATION="sum(p_j) by positive-mass plan",
         EXPECTED_RELATIONSHIP="n/a", BUSINESS_RATIONALE="Validation check only: normalized total must equal 1",
         TEMPORAL_AVAILABILITY="same unknown snapshot date as source", LEAKAGE_RISK="ZERO-VARIANCE / diagnostic only",
         MISSINGNESS_RISK="undefined for zero-mass plans", CARDINALITY_RISK="ZERO VARIANCE (=1)", REDUNDANCY_RISK="n/a",
         PRIORITY="n/a", STATUS="DIAGNOSTIC -- excluded from X"),
    dict(FEATURE_NAME="MARKET_SHARE_MAX", SOURCE_DATASET="Non-Hospital Market Share (notebook 14)", SOURCE_COLUMN="MARKET_SHARE_PCT",
         FEATURE_GROUP="D_CONCENTRATION", DESCRIPTION="Largest normalized provider weight",
         ORIGINAL_GRANULARITY="operator-plan-provider row", MODEL_GRANULARITY="plan; optional equal-complete-plan operator scenario", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="p_j = share_j / sum(share)", AGGREGATION="max(p_j); exclude count-incomplete plans from operator scenario",
         EXPECTED_RELATIONSHIP="?", BUSINESS_RATIONALE="Dependence on the largest observed provider",
         TEMPORAL_AVAILABILITY="unknown; source must be dated and available before cutoff", LEAKAGE_RISK="HIGH until dated/basis known",
         MISSINGNESS_RISK="undefined for zero-mass plans", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="Spearman rho=0.961 with HHI and -0.899 with provider count in complete plans",
         PRIORITY="MEDIUM", STATUS="PENDING -- excluded from X until linkage/date/scope/cardinality gates pass"),
    dict(FEATURE_NAME="MARKET_SHARE_TOP3", SOURCE_DATASET="Non-Hospital Market Share (notebook 14)", SOURCE_COLUMN="MARKET_SHARE_PCT",
         FEATURE_GROUP="D_CONCENTRATION", DESCRIPTION="Combined normalized weight of the three largest providers",
         ORIGINAL_GRANULARITY="operator-plan-provider row", MODEL_GRANULARITY="plan; optional equal-complete-plan operator scenario", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="p_j = share_j / sum(share)", AGGREGATION="sum(largest 3 p_j); exclude count-incomplete plans from operator scenario",
         EXPECTED_RELATIONSHIP="?", BUSINESS_RATIONALE="Top-three provider concentration",
         TEMPORAL_AVAILABILITY="unknown; source must be dated and available before cutoff", LEAKAGE_RISK="HIGH until dated/basis known",
         MISSINGNESS_RISK="undefined for zero-mass plans", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="same size/concentration signal family; screen jointly with count, MAX, TOP5 and HHI",
         PRIORITY="MEDIUM", STATUS="PENDING -- excluded from X until linkage/date/scope/cardinality gates pass"),
    dict(FEATURE_NAME="MARKET_SHARE_TOP5", SOURCE_DATASET="Non-Hospital Market Share (notebook 14)", SOURCE_COLUMN="MARKET_SHARE_PCT",
         FEATURE_GROUP="D_CONCENTRATION", DESCRIPTION="Combined normalized weight of the five largest providers",
         ORIGINAL_GRANULARITY="operator-plan-provider row", MODEL_GRANULARITY="plan; optional equal-complete-plan operator scenario", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="p_j = share_j / sum(share)", AGGREGATION="sum(largest 5 p_j); exclude count-incomplete plans from operator scenario",
         EXPECTED_RELATIONSHIP="?", BUSINESS_RATIONALE="Top-five provider concentration",
         TEMPORAL_AVAILABILITY="unknown; source must be dated and available before cutoff", LEAKAGE_RISK="HIGH until dated/basis known",
         MISSINGNESS_RISK="undefined for zero-mass plans", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="same size/concentration signal family; screen jointly with count, MAX, TOP3 and HHI",
         PRIORITY="MEDIUM", STATUS="PENDING -- excluded from X until linkage/date/scope/cardinality gates pass"),
    dict(FEATURE_NAME="HHI_NETWORK", SOURCE_DATASET="Non-Hospital Market Share (notebook 14)", SOURCE_COLUMN="MARKET_SHARE_PCT",
         FEATURE_GROUP="D_CONCENTRATION", DESCRIPTION="Herfindahl-Hirschman concentration of normalized provider weights on a 0-1 scale",
         ORIGINAL_GRANULARITY="operator-plan-provider row", MODEL_GRANULARITY="plan; optional equal-complete-plan operator scenario", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="p_j = share_j / sum(share)", AGGREGATION="sum(p_j**2); exclude count-incomplete plans from operator scenario",
         EXPECTED_RELATIONSHIP="?", BUSINESS_RATIONALE="Overall provider-network concentration",
         TEMPORAL_AVAILABILITY="unknown; source must be dated and available before cutoff", LEAKAGE_RISK="HIGH until dated/basis known",
         MISSINGNESS_RISK="undefined for zero-mass plans", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="Spearman rho=-0.947 with provider count and 0.961 with MAX in complete plans",
         PRIORITY="MEDIUM", STATUS="PENDING -- excluded from X until linkage/date/scope/cardinality gates pass"),
    dict(FEATURE_NAME="VERTICALIZED_PROVIDER", SOURCE_DATASET="future dated ownership/control crosswalk; absent", SOURCE_COLUMN="n/a",
         FEATURE_GROUP="D_VERTICALIZATION", DESCRIPTION="Operator-provider-date common-control label", ORIGINAL_GRANULARITY="operator-provider-date",
         MODEL_GRANULARITY="operator/plan at prediction date", RAW_OR_DERIVED="future evidence-backed label", TRANSFORMATION="1/0 only from verified positive/negative evidence; otherwise NA",
         AGGREGATION="none until dated entity linkage exists", EXPECTED_RELATIONSHIP="n/a", BUSINESS_RATIONALE="Ownership is distinct from network membership",
         TEMPORAL_AVAILABILITY="unavailable", LEAKAGE_RISK="NOT ASSESSABLE", MISSINGNESS_RISK="100% unknown", CARDINALITY_RISK="LOW",
         REDUNDANCY_RISK="n/a", PRIORITY="n/a", STATUS="PENDING EVIDENCE -- excluded from X; never map Unknown to 0"),
    dict(FEATURE_NAME="VERTICALIZED_OPERATOR", SOURCE_DATASET="future dated ownership/control crosswalk; absent", SOURCE_COLUMN="n/a",
         FEATURE_GROUP="D_VERTICALIZATION", DESCRIPTION="Operator-level verified common-control indicator", ORIGINAL_GRANULARITY="operator-date",
         MODEL_GRANULARITY="operator at prediction date", RAW_OR_DERIVED="future evidence-backed label", TRANSFORMATION="any verified owned provider; 0 requires explicit valid negative or complete assessment",
         AGGREGATION="operator-date after complete scoped assessment", EXPECTED_RELATIONSHIP="n/a", BUSINESS_RATIONALE="Potential ownership context, not inferable from concentration",
         TEMPORAL_AVAILABILITY="unavailable", LEAKAGE_RISK="NOT ASSESSABLE", MISSINGNESS_RISK="100% unknown", CARDINALITY_RISK="LOW",
         REDUNDANCY_RISK="n/a", PRIORITY="n/a", STATUS="PENDING EVIDENCE -- excluded from X; never map Unknown to 0"),
    dict(FEATURE_NAME="VERTICALIZATION_INTENSITY", SOURCE_DATASET="future dated ownership/control crosswalk; absent", SOURCE_COLUMN="n/a",
         FEATURE_GROUP="D_VERTICALIZATION", DESCRIPTION="Share of verified owned providers in a complete dated in-scope network", ORIGINAL_GRANULARITY="operator-date",
         MODEL_GRANULARITY="operator at prediction date", RAW_OR_DERIVED="future derived", TRANSFORMATION="verified owned providers / complete dated in-scope provider universe",
         AGGREGATION="operator-date", EXPECTED_RELATIONSHIP="n/a", BUSINESS_RATIONALE="Ownership intensity requires a verified numerator and denominator",
         TEMPORAL_AVAILABILITY="unavailable", LEAKAGE_RISK="NOT ASSESSABLE", MISSINGNESS_RISK="100% unknown", CARDINALITY_RISK="LOW",
         REDUNDANCY_RISK="n/a", PRIORITY="n/a", STATUS="PENDING EVIDENCE -- excluded from X; never map Unknown to 0"),
    dict(FEATURE_NAME="VERTICALIZED_MARKET_SHARE", SOURCE_DATASET="future dated ownership/control crosswalk plus validated allocation source; absent", SOURCE_COLUMN="n/a",
         FEATURE_GROUP="D_VERTICALIZATION", DESCRIPTION="Validated allocation mass assigned to verified owned providers", ORIGINAL_GRANULARITY="operator-provider-date",
         MODEL_GRANULARITY="operator at prediction date", RAW_OR_DERIVED="future derived", TRANSFORMATION="owned-provider validated mass / total validated mass",
         AGGREGATION="operator-date", EXPECTED_RELATIONSHIP="n/a", BUSINESS_RATIONALE="Weighted ownership intensity, only after both ownership and weights are validated",
         TEMPORAL_AVAILABILITY="unavailable", LEAKAGE_RISK="NOT ASSESSABLE", MISSINGNESS_RISK="100% unknown", CARDINALITY_RISK="LOW",
         REDUNDANCY_RISK="n/a", PRIORITY="n/a", STATUS="PENDING EVIDENCE -- excluded from X; never map Unknown to 0"),
    dict(FEATURE_NAME="VERTICALIZATION_TARGET", SOURCE_DATASET="notebook 14 audit candidate", SOURCE_COLUMN="VERTICALIZATION_TARGET",
         FEATURE_GROUP="D_VERTICALIZATION", DESCRIPTION="Evidence-backed Verticalized / Non-verticalized / Unknown state", ORIGINAL_GRANULARITY="operator-date",
         MODEL_GRANULARITY="operator at prediction date", RAW_OR_DERIVED="candidate label", TRANSFORMATION="preserve Unknown as Unknown",
         AGGREGATION="none", EXPECTED_RELATIONSHIP="n/a", BUSINESS_RATIONALE="Audit conclusion is Unknown; concentration cannot supply this target",
         TEMPORAL_AVAILABILITY="unavailable without dated ownership evidence", LEAKAGE_RISK="n/a", MISSINGNESS_RISK="100% Unknown in audited operators",
         CARDINALITY_RISK="LOW", REDUNDANCY_RISK="n/a", PRIORITY="n/a", STATUS="PENDING EVIDENCE -- excluded from X; never map Unknown to 0"),

    # --- E. ACTIVITY ---
    dict(FEATURE_NAME="N_PROCEDIMENTOS_DISTINTOS_TRAILING", SOURCE_DATASET="df_norte (re-engineered)", SOURCE_COLUMN="CD_PROCEDIMENTO",
         FEATURE_GROUP="E_ACTIVITY", DESCRIPTION="Distinct procedures observed in a trailing historical window",
         ORIGINAL_GRANULARITY="event", MODEL_GRANULARITY="operator-month (trailing)", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="nunique over rows dated before the prediction month", AGGREGATION="nunique by ID_PLANO and cutoff", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Historical breadth of care activity may explain future demand",
         TEMPORAL_AVAILABILITY="strictly historical window ending before prediction month; not yet engineered",
         LEAKAGE_RISK="MEDIUM if trailing / HIGH if full-year", MISSINGNESS_RISK="unknown until engineered",
         CARDINALITY_RISK="LOW", REDUNDANCY_RISK="may overlap with historical event volume",
         PRIORITY="MEDIUM", STATUS="CANDIDATE -- requires historical-window engineering"),
    dict(FEATURE_NAME="total_eventos (nb09/11, as-is)", SOURCE_DATASET="activity_matrix (nb09) / activity_matrix_by_municipality (nb11)", SOURCE_COLUMN="total_eventos",
         FEATURE_GROUP="E_ACTIVITY", DESCRIPTION="Annual (full 2025) event-volume aggregate",
         ORIGINAL_GRANULARITY="operator (full year)", MODEL_GRANULARITY="operator", RAW_OR_DERIVED="RAW (inherited from K-Means matrices)",
         TRANSFORMATION="none (raw counts)", AGGREGATION="none", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Larger operators -> more volume", TEMPORAL_AVAILABILITY="ANNUAL -- covers months after any given target MES",
         LEAKAGE_RISK="HIGH", MISSINGNESS_RISK="0%", CARDINALITY_RISK="n/a", REDUNDANCY_RISK="high with DEMAND_MEAN_6M",
         PRIORITY="n/a", STATUS="EXCLUDE in current form"),
    dict(FEATURE_NAME="eventos_por_mes_ativo", SOURCE_DATASET="activity_matrix (nb09)", SOURCE_COLUMN="eventos_por_mes_ativo",
         FEATURE_GROUP="E_ACTIVITY", DESCRIPTION="Average events per active month, annual",
         ORIGINAL_GRANULARITY="operator (full year)", MODEL_GRANULARITY="operator", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="ratio", AGGREGATION="none", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Activity intensity", TEMPORAL_AVAILABILITY="ANNUAL (leak)", LEAKAGE_RISK="HIGH",
         MISSINGNESS_RISK="0%", CARDINALITY_RISK="n/a", REDUNDANCY_RISK="r=0.999 with total_eventos, already documented in nb09",
         PRIORITY="n/a", STATUS="EXCLUDE (redundant + leak)"),
    dict(FEATURE_NAME="n_municipality_pairs_total", SOURCE_DATASET="activity_matrix_by_municipality (nb11)", SOURCE_COLUMN="groupby size",
         FEATURE_GROUP="E_ACTIVITY", DESCRIPTION="Distinct (beneficiary muni x provider muni) combinations for the operator",
         ORIGINAL_GRANULARITY="operator-month (cumulative, as of)", MODEL_GRANULARITY="operator", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="count", AGGREGATION="count", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Broader geographic combination coverage", TEMPORAL_AVAILABILITY="as-of cumulative value in the analytical table (prior claims only); the nb11/12 annual files would leak; a trailing window is still to be engineered",
         LEAKAGE_RISK="LOW as-of / HIGH if the nb11/12 annual file is reused", MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="high with municipios_beneficiarios/prestadores (nb09)",
         PRIORITY="LOW in current form", STATUS="CANDIDATE -- requires trailing re-engineering"),
    dict(FEATURE_NAME="n_municipality_pairs_active", SOURCE_DATASET="active_operator_municipality_pairs (nb12)", SOURCE_COLUMN="groupby size",
         FEATURE_GROUP="E_ACTIVITY", DESCRIPTION="Distinct municipality pairs classified ACTIVE by nb12's K-Means",
         ORIGINAL_GRANULARITY="operator-month (cumulative, as of)", MODEL_GRANULARITY="operator", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="count", AGGREGATION="count", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Robust (not one-off) geographic presence count", TEMPORAL_AVAILABILITY="as-of cumulative value in the analytical table (prior claims only); the nb11/12 annual files would leak; a trailing window is still to be engineered",
         LEAKAGE_RISK="LOW as-of / HIGH if the nb11/12 annual file is reused", MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="high with STATUS_ATUACAO (its own input)",
         PRIORITY="LOW in current form", STATUS="CANDIDATE -- requires trailing re-engineering"),
    dict(FEATURE_NAME="share_active_municipality_pairs", SOURCE_DATASET="derived (nb11 + nb12 rollup)", SOURCE_COLUMN="ratio of the two above",
         FEATURE_GROUP="E_ACTIVITY", DESCRIPTION="Share of the operator's municipality pairs that are 'active'",
         ORIGINAL_GRANULARITY="operator-month (cumulative, as of)", MODEL_GRANULARITY="operator", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="ratio", AGGREGATION="none", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Concentration vs. dispersion of robust activity in geographic space", TEMPORAL_AVAILABILITY="as-of cumulative value in the analytical table (prior claims only); the nb11/12 annual files would leak; a trailing window is still to be engineered",
         LEAKAGE_RISK="LOW as-of / HIGH if the nb11/12 annual file is reused", MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW", REDUNDANCY_RISK="high with the two above",
         PRIORITY="LOW in current form", STATUS="CANDIDATE -- requires trailing re-engineering"),

    # --- G. TEMPORAL ---
    dict(FEATURE_NAME="MONTH", SOURCE_DATASET="modeling_matrix_v1", SOURCE_COLUMN="MES", FEATURE_GROUP="G_TEMPORAL",
         DESCRIPTION="Calendar month (1-12)", ORIGINAL_GRANULARITY="month", MODEL_GRANULARITY="month", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION=".dt.month", AGGREGATION="none", EXPECTED_RELATIONSHIP="?",
         BUSINESS_RATIONALE="Seasonality documented in project docs (Aug-Sep dip, Nov-Dec rise / Pink October)",
         TEMPORAL_AVAILABILITY="known with full advance certainty", LEAKAGE_RISK="LOW", MISSINGNESS_RISK="0%",
         CARDINALITY_RISK="LOW (12)", REDUNDANCY_RISK="low", PRIORITY="HIGH", STATUS="APPROVED"),
    dict(FEATURE_NAME="QUARTER", SOURCE_DATASET="modeling_matrix_v1", SOURCE_COLUMN="MES", FEATURE_GROUP="G_TEMPORAL",
         DESCRIPTION="Calendar quarter", ORIGINAL_GRANULARITY="month", MODEL_GRANULARITY="month", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION=".dt.quarter", AGGREGATION="none", EXPECTED_RELATIONSHIP="?",
         BUSINESS_RATIONALE="Coarser seasonality", TEMPORAL_AVAILABILITY="known in advance", LEAKAGE_RISK="LOW",
         MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW (4)", REDUNDANCY_RISK="high with MONTH",
         PRIORITY="LOW (redundant)", STATUS="CANDIDATE"),
    dict(FEATURE_NAME="SEASONALITY (Pink October flag)", SOURCE_DATASET="modeling_matrix_v1", SOURCE_COLUMN="MES", FEATURE_GROUP="G_TEMPORAL",
         DESCRIPTION="Binary flag for Oct/Nov", ORIGINAL_GRANULARITY="month", MODEL_GRANULARITY="month", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="MONTH.isin([10,11])", AGGREGATION="none", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Documented effect in project docs (Sec. 8.1)", TEMPORAL_AVAILABILITY="known in advance",
         LEAKAGE_RISK="LOW", MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW (2)", REDUNDANCY_RISK="high with MONTH",
         PRIORITY="MEDIUM", STATUS="CANDIDATE"),
    dict(FEATURE_NAME="YEAR", SOURCE_DATASET="modeling_matrix_v1", SOURCE_COLUMN="MES", FEATURE_GROUP="G_TEMPORAL",
         DESCRIPTION="Calendar year", ORIGINAL_GRANULARITY="month", MODEL_GRANULARITY="month", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION=".dt.year", AGGREGATION="none", EXPECTED_RELATIONSHIP="n/a",
         BUSINESS_RATIONALE="n/a", TEMPORAL_AVAILABILITY="n/a", LEAKAGE_RISK="LOW", MISSINGNESS_RISK="0%",
         CARDINALITY_RISK="ZERO VARIANCE (only 2025 available)", REDUNDANCY_RISK="n/a",
         PRIORITY="n/a", STATUS="EXCLUDE -- zero variance"),

    # --- H. HISTORICAL DEMAND ---
    dict(FEATURE_NAME="DEMAND_LAG_1M", SOURCE_DATASET="modeling_matrix_v1", SOURCE_COLUMN="volume_demanda_proxy", FEATURE_GROUP="H_HISTORICAL",
         DESCRIPTION="Target value, same group, 1 month before", ORIGINAL_GRANULARITY="group-month", MODEL_GRANULARITY="group-month",
         RAW_OR_DERIVED="DERIVED", TRANSFORMATION="shift(1) within group", AGGREGATION="none", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Month-to-month demand autocorrelation", TEMPORAL_AVAILABILITY="strictly prior to target",
         LEAKAGE_RISK="LOW", MISSINGNESS_RISK="first group month plus any unusable bundled-history month", CARDINALITY_RISK="n/a",
         REDUNDANCY_RISK="high with LAG_3M/6M", PRIORITY="HIGH", STATUS="APPROVED"),
    dict(FEATURE_NAME="DEMAND_LAG_3M", SOURCE_DATASET="modeling_matrix_v1", SOURCE_COLUMN="volume_demanda_proxy", FEATURE_GROUP="H_HISTORICAL",
         DESCRIPTION="Target, 3 months before", ORIGINAL_GRANULARITY="group-month", MODEL_GRANULARITY="group-month",
         RAW_OR_DERIVED="DERIVED", TRANSFORMATION="shift(3)", AGGREGATION="none", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Quarterly pattern", TEMPORAL_AVAILABILITY="prior", LEAKAGE_RISK="LOW",
         MISSINGNESS_RISK="first 3 group months plus unusable bundled history", CARDINALITY_RISK="n/a", REDUNDANCY_RISK="medium", PRIORITY="HIGH", STATUS="APPROVED"),
    dict(FEATURE_NAME="DEMAND_LAG_6M", SOURCE_DATASET="modeling_matrix_v1", SOURCE_COLUMN="volume_demanda_proxy", FEATURE_GROUP="H_HISTORICAL",
         DESCRIPTION="Target, 6 months before", ORIGINAL_GRANULARITY="group-month", MODEL_GRANULARITY="group-month",
         RAW_OR_DERIVED="DERIVED", TRANSFORMATION="shift(6)", AGGREGATION="none", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Semiannual pattern", TEMPORAL_AVAILABILITY="prior", LEAKAGE_RISK="LOW",
         MISSINGNESS_RISK="first 6 group months plus unusable bundled history", CARDINALITY_RISK="n/a", REDUNDANCY_RISK="medium", PRIORITY="MEDIUM", STATUS="APPROVED"),
    dict(FEATURE_NAME="DEMAND_MEAN_3M", SOURCE_DATASET="modeling_matrix_v1", SOURCE_COLUMN="volume_demanda_proxy", FEATURE_GROUP="H_HISTORICAL",
         DESCRIPTION="3-month rolling mean, computed on already-lagged values", ORIGINAL_GRANULARITY="group-month",
         MODEL_GRANULARITY="group-month", RAW_OR_DERIVED="DERIVED", TRANSFORMATION="shift(1).rolling(3).mean()",
         AGGREGATION="mean", EXPECTED_RELATIONSHIP="+", BUSINESS_RATIONALE="Smooths month-to-month noise",
         TEMPORAL_AVAILABILITY="prior (entire window in the past)", LEAKAGE_RISK="LOW", MISSINGNESS_RISK="initial history plus any unknown/incomplete target in the window",
         CARDINALITY_RISK="n/a", REDUNDANCY_RISK="high with LAG_1M", PRIORITY="HIGH", STATUS="APPROVED"),
    dict(FEATURE_NAME="DEMAND_MEAN_6M", SOURCE_DATASET="modeling_matrix_v1", SOURCE_COLUMN="volume_demanda_proxy", FEATURE_GROUP="H_HISTORICAL",
         DESCRIPTION="6-month rolling mean, lagged", ORIGINAL_GRANULARITY="group-month", MODEL_GRANULARITY="group-month",
         RAW_OR_DERIVED="DERIVED", TRANSFORMATION="shift(1).rolling(6).mean()", AGGREGATION="mean",
         EXPECTED_RELATIONSHIP="+", BUSINESS_RATIONALE="Medium-term trend", TEMPORAL_AVAILABILITY="prior",
         LEAKAGE_RISK="LOW", MISSINGNESS_RISK="initial history plus any unknown/incomplete target in the window", CARDINALITY_RISK="n/a", REDUNDANCY_RISK="high with MEAN_3M",
         PRIORITY="MEDIUM", STATUS="APPROVED"),
    dict(FEATURE_NAME="DEMAND_STD_3M", SOURCE_DATASET="modeling_matrix_v1", SOURCE_COLUMN="volume_demanda_proxy", FEATURE_GROUP="H_HISTORICAL",
         DESCRIPTION="Recent demand volatility, lagged", ORIGINAL_GRANULARITY="group-month", MODEL_GRANULARITY="group-month",
         RAW_OR_DERIVED="DERIVED", TRANSFORMATION="shift(1).rolling(3).std()", AGGREGATION="std",
         EXPECTED_RELATIONSHIP="?", BUSINESS_RATIONALE="Volatile operators may be harder to predict -- uncertainty feature",
         TEMPORAL_AVAILABILITY="prior", LEAKAGE_RISK="LOW", MISSINGNESS_RISK="at least two prior valid months plus no incomplete target in the window", CARDINALITY_RISK="n/a",
         REDUNDANCY_RISK="low", PRIORITY="MEDIUM", STATUS="APPROVED"),
    dict(FEATURE_NAME="DEMAND_GROWTH_MOM", SOURCE_DATASET="modeling_matrix_v1", SOURCE_COLUMN="volume_demanda_proxy", FEATURE_GROUP="H_HISTORICAL",
         DESCRIPTION="Month-over-month growth (lag1 vs lag2)", ORIGINAL_GRANULARITY="group-month", MODEL_GRANULARITY="group-month",
         RAW_OR_DERIVED="DERIVED", TRANSFORMATION="(lag1-lag2)/lag2", AGGREGATION="none", EXPECTED_RELATIONSHIP="?",
         BUSINESS_RATIONALE="Growth/decline momentum", TEMPORAL_AVAILABILITY="prior", LEAKAGE_RISK="LOW",
         MISSINGNESS_RISK="undefined for zero lag2, insufficient history or unusable bundled targets", CARDINALITY_RISK="n/a", REDUNDANCY_RISK="low",
         PRIORITY="LOW (very sparse)", STATUS="CANDIDATE"),
    dict(FEATURE_NAME="DEMAND_GROWTH_YOY", SOURCE_DATASET="n/a", SOURCE_COLUMN="n/a", FEATURE_GROUP="H_HISTORICAL",
         DESCRIPTION="Year-over-year growth", ORIGINAL_GRANULARITY="n/a", MODEL_GRANULARITY="n/a", RAW_OR_DERIVED="n/a",
         TRANSFORMATION="n/a", AGGREGATION="n/a", EXPECTED_RELATIONSHIP="n/a",
         BUSINESS_RATIONALE="Not computable -- only 2025 available", TEMPORAL_AVAILABILITY="n/a", LEAKAGE_RISK="n/a",
         MISSINGNESS_RISK="n/a", CARDINALITY_RISK="n/a", REDUNDANCY_RISK="n/a", PRIORITY="n/a",
         STATUS="EXCLUDE -- insufficient history (1 year only)"),

    # --- STATUS_ATUACAO ---
    dict(FEATURE_NAME="STATUS_ATUACAO", SOURCE_DATASET="active_operator_municipality_pairs (nb12), rolled up", SOURCE_COLUMN="derived",
         FEATURE_GROUP="CONTEXT", DESCRIPTION="ATUANTE/MARGINAL as of the row's month: operator has >=1 active municipality pair on prior claims",
         ORIGINAL_GRANULARITY="operator-month (as of)", MODEL_GRANULARITY="operator-month", RAW_OR_DERIVED="DERIVED",
         TRANSFORMATION="nb11/12 method (k=2) re-applied to claims before each month + rollup + threshold", AGGREGATION="any(active)", EXPECTED_RELATIONSHIP="+",
         BUSINESS_RATIONALE="Population-selection context; not a target substitute or predictor",
         TEMPORAL_AVAILABILITY="as of each target month, prior claims only; the full-year version is kept for Section 4.2 and never selects rows; definition disagrees 13% with nb10 (Section 4.2)",
         LEAKAGE_RISK="LOW as applied (as-of)", MISSINGNESS_RISK="0%", CARDINALITY_RISK="LOW (2)",
         REDUNDANCY_RISK="depends on which segmentation is chosen as ground truth",
         PRIORITY="HIGH as a population filter / n/a as a predictor", STATUS="POPULATION FILTER -- excluded from X"),
])

pending_network_features = {
    "N_PRESTADORES_NO_GRUPO", "N_PRESTADORES", "MARKET_SHARE_MAX",
    "MARKET_SHARE_TOP3", "MARKET_SHARE_TOP5", "HHI_NETWORK",
}
verticalization_fields = {
    "VERTICALIZED_PROVIDER", "VERTICALIZED_OPERATOR", "VERTICALIZATION_INTENSITY",
    "VERTICALIZED_MARKET_SHARE", "VERTICALIZATION_TARGET",
}
proposal_status = feature_proposal_matrix.set_index("FEATURE_NAME")["STATUS"]
approved_features = set(proposal_status[proposal_status.str.startswith("APPROVED")].index)
assert proposal_status.loc[list(pending_network_features)].str.startswith("PENDING").all()
assert proposal_status.loc[list(verticalization_fields)].str.contains("excluded from X").all()
assert proposal_status.loc["MARKET_SHARE_TOTAL"] == "DIAGNOSTIC -- excluded from X"
assert proposal_status.loc["STATUS_ATUACAO"] == "POPULATION FILTER -- excluded from X"

print("feature_proposal_matrix shape:", feature_proposal_matrix.shape)
feature_proposal_matrix

feature_proposal_matrix shape: (40, 19)


,FEATURE_NAME,SOURCE_DATASET,SOURCE_COLUMN,FEATURE_GROUP,DESCRIPTION,ORIGINAL_GRANULARITY,MODEL_GRANULARITY,RAW_OR_DERIVED,TRANSFORMATION,AGGREGATION,EXPECTED_RELATIONSHIP,BUSINESS_RATIONALE,TEMPORAL_AVAILABILITY,LEAKAGE_RISK,MISSINGNESS_RISK,CARDINALITY_RISK,REDUNDANCY_RISK,PRIORITY,STATUS
0,PORTE,df_norte,PORTE,A_OPERATOR_PLAN,Operator size tier by total active beneficiaries,operator,operator (static),RAW,none (ordinal already encoded as PORTE_ORDINAL),none,+,Larger operators tend to larger absolute volume,pre-period (confirmed time-invariant per opera...,LOW,0%,LOW (4 levels),"low (not redundant with STATUS_ATUACAO, checked)",HIGH,APPROVED
1,PORTE_ORDINAL,df_norte,PORTE_ORDINAL,A_OPERATOR_PLAN,Ordinal-encoded version of PORTE,operator,operator (static),RAW,none,none,+,Same as PORTE,pre-period,LOW,0%,LOW,"high with PORTE (same info, different format)",HIGH,"APPROVED (use one of the two, not both)"
2,NM_MODALIDADE_*,df_norte,NM_MODALIDADE_* (6 one-hot cols),A_OPERATOR_PLAN,Operator's institutional modality,operator,operator (static),RAW (already one-hot),none,none,?,Modality may correlate with network strategy,pre-period,LOW,0%,LOW,low,MEDIUM,CANDIDATE
3,ID_PLANO,df_norte,ID_PLANO,A_OPERATOR_PLAN,Operator identifier,operator,operator (key),RAW,NONE -- not numeric,none,n/a,"It is the key, not a feature",n/a,n/a,0%,HIGH (3289),n/a,n/a,EXCLUDE (as numeric feature; use as key/group ...
4,PCT_FEMININO,df_norte,SEXO_Feminino,B_DEMOGRAPHIC,Share of group's rows that are female,event,group (contemporaneous),DERIVED,proportion,mean,?,Sex composition may vary by treatment line,contemporaneous to target,HIGH,0%,LOW,low,LOW now / MEDIUM if lagged,CANDIDATE (only as a trailing/lagged version)
5,PCT_AGE_60PLUS,df_norte,FAIXA_ETARIA_ORDINAL,B_DEMOGRAPHIC,Share of group's rows in age >= 60,event,group (contemporaneous),DERIVED,threshold + proportion,mean,+,Breast cancer incidence concentrates in older ...,contemporaneous,HIGH,0%,LOW,low,LOW now / MEDIUM if lagged,CANDIDATE (only as a trailing/lagged version)
6,UF_PRESTADOR,df_norte,UF_PRESTADOR,C_GEOGRAPHIC,State where the service was provided,event,already a modeling key,RAW,none,none,n/a,Already the model's geographic key,pre-determined,LOW,0%,LOW (7),n/a (is a key),n/a,"KEY, not a feature to select"
7,N_MUNICIPIOS_BENEFICIARIO_TRAILING,df_norte (re-engineered),CD_MUNICIPIO_BENEFICIARIO,C_GEOGRAPHIC,"Distinct beneficiary-residence municipalities,...",operator-month (trailing),operator (trailing),DERIVED,"nunique over a rolling window, NOT the nb08 an...",nunique,+,Broader demand-side geographic reach,strictly historical window ending before predi...,MEDIUM (if trailing) / HIGH (if reusing nb08/1...,0%,LOW,low,MEDIUM,"CANDIDATE -- requires new engineering, not dir..."
8,N_MUNICIPIOS_PRESTADOR_TRAILING,df_norte (re-engineered),CD_MUNICIPIO_PRESTADOR,C_GEOGRAPHIC,"Distinct provider municipalities, trailing window",operator-month (trailing),operator (trailing),DERIVED,nunique over a rolling window,nunique,+,Broader provider-side geographic reach,strictly historical window ending before predi...,MEDIUM (if trailing) / HIGH (as-is),0%,LOW,low,MEDIUM,CANDIDATE -- requires historical-window engine...
9,N_PRESTADORES_NO_GRUPO,Non-Hospital Market Share (notebook 13),N_PRESTADORES_NO_GRUPO,D_NETWORK,Declared provider count repeated within a plan...,operator-plan-provider row; constant within plan,plan snapshot; no valid scalar operator roll-up,RAW,first after asserting one value per plan,retain by plan; count-incomplete plans remain ...,?,Plan-level network size may explain access cap...,unknown source reference/publication date; mus...,HIGH until dated,1 of 46 audited plans is count-incomplete,LOW,same as observed count in 45 of 46 plans,MEDIUM,PENDING -- excluded from X until linkage/date/...


## 6. Historical Demand Features

Lag and rolling features must use **only** information strictly before the target
period. Each `(ID_PLANO, treatment_line_primary, UF_PRESTADOR)` combination is reindexed
from its operator's **first** observed claim month to the **last month of the data**, and
every month without a claim in that span is a zero of billed volume.

**Why the panel runs to the end of the data.** Stopping each operator's panel at its
*last* observed claim month would itself use future information: at any forecast date
nobody knows which claim will turn out to be an operator's last, so the months after an
operator's final claim would be silently missing from the data — exactly the months in
which a forecaster would have had to predict an exit. Running the panel to the end of
the data removes that. The zeros remain claims-based: a month without a claim is zero *billed*
volume, not proof of zero demand or of enrollment.

The panel is built for **every** operator, not only ATUANTE ones, and the lags are
computed on it before any row is filtered out (Section 6.1), so the eligibility filter
never breaks a row's history.

Observed groups affected by bundling remain distinguishable from zero rows. Fully
bundled targets stay null, and partially bundled sums are retained only as diagnostics.
Neither type is used as a training target or as historical input to lag/rolling features.


In [11]:
GROUP_KEYS_NO_MONTH = ["ID_PLANO", "treatment_line_primary", "UF_PRESTADOR"]

combos = target[GROUP_KEYS_NO_MONTH].drop_duplicates()
operator_first = (target.groupby("ID_PLANO", observed=True)["MES"].min()
                  .rename("OPERATOR_FIRST_MONTH").reset_index())
combo_first = (target.groupby(GROUP_KEYS_NO_MONTH, observed=True)["MES"].min()
               .rename("COMBO_FIRST_MONTH").reset_index())
full_index = (
    combos.merge(pd.DataFrame({"MES": FULL_MONTHS}), how="cross")
    .merge(operator_first, on="ID_PLANO", how="left", validate="many_to_one")
)
# From the operator's first claim to the end of the data: no end point taken from the future.
full_index = full_index.loc[full_index["MES"] >= full_index["OPERATOR_FIRST_MONTH"]].drop(
    columns="OPERATOR_FIRST_MONTH")
print("Distinct (operator x treatment_line x UF) combos, all operators:", len(combos))
print("Reindexed panel rows (operator's first claim -> end of data):", len(full_index))

panel = full_index.merge(
    target[GROUP_KEYS_NO_MONTH + ["MES", "volume_demanda_proxy", "N_SOURCE_ROWS",
                   "N_BUNDLED_ROWS", "BUNDLING_RATE", "TARGET_MEASUREMENT_STATUS",
                   "TARGET_USABLE_FOR_TRAINING"]],
    on=GROUP_KEYS_NO_MONTH + ["MES"], how="left", indicator="_target_merge",
    validate="one_to_one",
)
structural_zero = panel["_target_merge"].eq("left_only")
observed_unknown = panel["_target_merge"].eq("both") & panel["volume_demanda_proxy"].isna()
panel.loc[structural_zero, "volume_demanda_proxy"] = 0
panel.loc[structural_zero, ["N_SOURCE_ROWS", "N_BUNDLED_ROWS"]] = 0
panel.loc[structural_zero, "TARGET_MEASUREMENT_STATUS"] = "STRUCTURAL_ZERO_NO_CLAIM"
panel.loc[structural_zero, "TARGET_USABLE_FOR_TRAINING"] = True
panel["IS_STRUCTURAL_ZERO"] = structural_zero

assert panel.loc[observed_unknown, "TARGET_MEASUREMENT_STATUS"].eq("OBSERVED_ALL_BUNDLED_UNKNOWN").all()
assert panel.loc[observed_unknown, "volume_demanda_proxy"].isna().all()
assert panel.loc[structural_zero, "volume_demanda_proxy"].eq(0).all()
panel = panel.drop(columns="_target_merge")

before = len(panel)
panel = panel.merge(combo_first, on=GROUP_KEYS_NO_MONTH, how="left", validate="many_to_one")
# As-of status of the row's own month; NaN where the operator had no prior claim (e.g. January).
panel = panel.merge(status_asof, on=["ID_PLANO", "MES"], how="left", validate="many_to_one")
assert len(panel) == before, "status merge must not duplicate panel rows"

panel = panel.sort_values(GROUP_KEYS_NO_MONTH + ["MES"]).reset_index(drop=True)

print("panel shape:", panel.shape)
print("Duplicated keys:", panel.duplicated(subset=GROUP_KEYS_NO_MONTH + ["MES"]).sum())
print("Target measurement status:")
print(panel["TARGET_MEASUREMENT_STATUS"].value_counts())


Distinct (operator x treatment_line x UF) combos, all operators: 2305
Reindexed panel rows (operator's first claim -> end of data): 24841


panel shape: (24841, 16)
Duplicated keys: 0
Target measurement status:
TARGET_MEASUREMENT_STATUS
STRUCTURAL_ZERO_NO_CLAIM        14539
OBSERVED_MEASURED               10148
OBSERVED_PARTIALLY_BUNDLED        128
OBSERVED_ALL_BUNDLED_UNKNOWN       26
Name: count, dtype: int64


In [12]:
panel["_history_volume"] = panel["volume_demanda_proxy"].where(panel["TARGET_USABLE_FOR_TRAINING"])
gkey = panel.groupby(GROUP_KEYS_NO_MONTH, observed=True)["_history_volume"]
panel["DEMAND_LAG_1M"] = gkey.shift(1)
panel["DEMAND_LAG_3M"] = gkey.shift(3)
panel["DEMAND_LAG_6M"] = gkey.shift(6)

panel["_shift1"] = gkey.shift(1)
grp_shift = panel.groupby(GROUP_KEYS_NO_MONTH, observed=True)["_shift1"]
panel["DEMAND_MEAN_3M"] = grp_shift.transform(lambda s: s.rolling(3, min_periods=1).mean())
panel["DEMAND_MEAN_6M"] = grp_shift.transform(lambda s: s.rolling(6, min_periods=1).mean())
panel["DEMAND_STD_3M"] = grp_shift.transform(lambda s: s.rolling(3, min_periods=2).std())

# If an earlier target was unknown or incomplete, do not hide it by letting rolling()
# silently skip that value. The affected historical window remains unavailable.
panel["_history_invalid"] = ~panel["TARGET_USABLE_FOR_TRAINING"].astype(bool)
panel["_history_invalid_shift1"] = (
    panel.groupby(GROUP_KEYS_NO_MONTH, observed=True)["_history_invalid"].shift(1).fillna(False)
)
grp_invalid = panel.groupby(GROUP_KEYS_NO_MONTH, observed=True)["_history_invalid_shift1"]
invalid_3m = grp_invalid.transform(lambda s: s.rolling(3, min_periods=1).max()).astype(bool)
invalid_6m = grp_invalid.transform(lambda s: s.rolling(6, min_periods=1).max()).astype(bool)
panel.loc[invalid_3m, ["DEMAND_MEAN_3M", "DEMAND_STD_3M"]] = np.nan
panel.loc[invalid_6m, "DEMAND_MEAN_6M"] = np.nan

panel["DEMAND_LAG_2M"] = gkey.shift(2)
panel["DEMAND_GROWTH_MOM"] = (panel["DEMAND_LAG_1M"] - panel["DEMAND_LAG_2M"]) / panel["DEMAND_LAG_2M"].replace(0, np.nan)
panel = panel.drop(columns=["_history_volume", "_shift1", "_history_invalid",
                            "_history_invalid_shift1", "DEMAND_LAG_2M"])

sample_id = panel["ID_PLANO"].iloc[0]
cols_show = GROUP_KEYS_NO_MONTH + ["MES", "volume_demanda_proxy", "TARGET_MEASUREMENT_STATUS",
                                    "BUNDLING_RATE", "DEMAND_LAG_1M", "DEMAND_LAG_3M", "DEMAND_MEAN_3M"]
panel[panel["ID_PLANO"] == sample_id][cols_show].head(9)

,ID_PLANO,treatment_line_primary,UF_PRESTADOR,MES,volume_demanda_proxy,TARGET_MEASUREMENT_STATUS,BUNDLING_RATE,DEMAND_LAG_1M,DEMAND_LAG_3M,DEMAND_MEAN_3M
0,1019993,adjuvant,TO,2025-01-01,0.0,STRUCTURAL_ZERO_NO_CLAIM,NaN,NaN,NaN,NaN
1,1019993,adjuvant,TO,2025-02-01,0.0,STRUCTURAL_ZERO_NO_CLAIM,NaN,0.0,NaN,0.000000
2,1019993,adjuvant,TO,2025-03-01,1.0,OBSERVED_MEASURED,0.0,0.0,NaN,0.000000
3,1019993,adjuvant,TO,2025-04-01,0.0,STRUCTURAL_ZERO_NO_CLAIM,NaN,1.0,0.0,0.333333
4,1019993,adjuvant,TO,2025-05-01,0.0,STRUCTURAL_ZERO_NO_CLAIM,NaN,0.0,0.0,0.333333
5,1019993,adjuvant,TO,2025-06-01,0.0,STRUCTURAL_ZERO_NO_CLAIM,NaN,0.0,1.0,0.333333
6,1019993,adjuvant,TO,2025-07-01,0.0,STRUCTURAL_ZERO_NO_CLAIM,NaN,0.0,0.0,0.000000
7,1019993,adjuvant,TO,2025-08-01,0.0,STRUCTURAL_ZERO_NO_CLAIM,NaN,0.0,0.0,0.000000
8,1019993,adjuvant,TO,2025-09-01,0.0,STRUCTURAL_ZERO_NO_CLAIM,NaN,0.0,0.0,0.000000


### 6.1 As-of eligibility — only rows a forecaster could have been asked for

A row for month *s* enters the modeling matrix only if, on the claims billed before *s*:

1. its operator was ATUANTE (Section 4.0), and
2. its `operator x line x UF` combination had already been billed at least once.

Condition 2 matters as much as condition 1. Without it, a line first billed in November
would be zero-filled back to the operator's first month, creating training rows that exist
only because of a future claim. With both conditions, every row is a forecast that someone
standing at the start of that month could have been asked to make, so **any split on `MES`
is free of look-ahead by construction** — the holdout and every validation fold alike, not
only one cutoff chosen in advance.

The filter runs after the lags, so it removes rows without touching any other row's history.

In [13]:
panel_complete = panel  # every operator, first claim -> end of data; kept for the counts below
is_atuante = panel_complete["STATUS_ATUACAO"].eq("ATUANTE")
is_known = panel_complete["MES"] > panel_complete["COMBO_FIRST_MONTH"]
panel = panel_complete.loc[is_atuante & is_known].reset_index(drop=True)

print(f"complete panel rows (all operators):                       {len(panel_complete):,}")
print(f"excluded -- operator not ATUANTE as of the month:           {int((~is_atuante).sum()):,}")
print(f"excluded -- ATUANTE, but combination not yet billed:        {int((is_atuante & ~is_known).sum()):,}")
print(f"eligible rows (ATUANTE as of the month, combination known): {len(panel):,}")
print(f"distinct operators with at least one eligible row:          {panel['ID_PLANO'].nunique():,}")
print("\nEligible rows per month:")
print(panel.groupby("MES").size().to_string())

complete panel rows (all operators):                       24,841
excluded -- operator not ATUANTE as of the month:           11,372
excluded -- ATUANTE, but combination not yet billed:        1,820
eligible rows (ATUANTE as of the month, combination known): 11,649
distinct operators with at least one eligible row:          507

Eligible rows per month:
MES
2025-02-01     567
2025-03-01     792
2025-04-01     927
2025-05-01     943
2025-06-01    1060
2025-07-01    1099
2025-08-01    1169
2025-09-01    1249
2025-10-01    1228
2025-11-01    1289
2025-12-01    1326


## 7. Modeling Matrix Assembly

Static operator attributes (`PORTE`, `PORTE_ORDINAL`) are merged by `ID_PLANO` only after
an explicit per-operator uniqueness check across the observed rows. Every merge is checked
against row-count growth and key duplication immediately after it runs.

`modeling_matrix_v1` is an auditable analytical table and therefore retains keys, target and
the as-of population-filter column (with the as-of municipality-pair counts behind it). `X_current` below is the actual predictor view: it deliberately
excludes `STATUS_ATUACAO`, all pending network/concentration candidates, the normalized-total
diagnostic and every verticalization field. Rows with unknown or partially measured
targets remain in `modeling_matrix_v1` for auditability but are excluded from `X_current`
and `y_current`.

In [14]:
porte_columns = ["PORTE", "PORTE_ORDINAL"]
porte_consistency = df_norte.groupby("ID_PLANO", observed=True)[porte_columns].nunique(dropna=False)
assert porte_consistency.max().eq(1).all(), "PORTE fields must be constant within each operator"

porte_static = df_norte.drop_duplicates("ID_PLANO")[["ID_PLANO"] + porte_columns]
assert porte_static["ID_PLANO"].duplicated().sum() == 0

before = len(panel)
modeling_matrix_v1 = panel.merge(porte_static, on="ID_PLANO", how="left")
assert len(modeling_matrix_v1) == before, "merge must not duplicate rows"

modeling_matrix_v1["MONTH"] = modeling_matrix_v1["MES"].dt.month
modeling_matrix_v1["QUARTER"] = modeling_matrix_v1["MES"].dt.quarter

FEATURE_ORDER = [
    "ID_PLANO", "MES", "UF_PRESTADOR", "treatment_line_primary", "STATUS_ATUACAO",
    "volume_demanda_proxy", "TARGET_MEASUREMENT_STATUS", "TARGET_USABLE_FOR_TRAINING",
    "IS_STRUCTURAL_ZERO", "N_SOURCE_ROWS", "N_BUNDLED_ROWS", "BUNDLING_RATE",
    "DEMAND_LAG_1M", "DEMAND_LAG_3M", "DEMAND_LAG_6M",
    "DEMAND_MEAN_3M", "DEMAND_MEAN_6M", "DEMAND_STD_3M", "DEMAND_GROWTH_MOM",
    "PORTE", "PORTE_ORDINAL", "MONTH", "QUARTER",
    "n_municipality_pairs_total", "n_municipality_pairs_active", "share_active_municipality_pairs",
]
modeling_matrix_v1 = modeling_matrix_v1[FEATURE_ORDER]

CURRENT_X_COLUMNS = [
    "DEMAND_LAG_1M", "DEMAND_LAG_3M", "DEMAND_LAG_6M",
    "DEMAND_MEAN_3M", "DEMAND_MEAN_6M", "DEMAND_STD_3M",
    "PORTE_ORDINAL", "MONTH",
]
assert set(CURRENT_X_COLUMNS) <= approved_features, "Every X column must be APPROVED in the proposal matrix"
training_mask = modeling_matrix_v1["TARGET_USABLE_FOR_TRAINING"].astype(bool)
X_current = modeling_matrix_v1.loc[training_mask, CURRENT_X_COLUMNS].copy()
y_current = modeling_matrix_v1.loc[training_mask, "volume_demanda_proxy"].copy()
assert X_current.index.equals(y_current.index)
assert y_current.notna().all()
assert y_current.name == "volume_demanda_proxy"

print("SHAPE:", modeling_matrix_v1.shape)
print("CURRENT X SHAPE:", X_current.shape)
print("CURRENT X COLUMNS:", X_current.columns.tolist())
print("ROWS EXCLUDED FOR UNKNOWN/PARTIAL TARGET:", int((~training_mask).sum()))
print("\nCOLUMNS:")
print(modeling_matrix_v1.columns.tolist())
print("\nDUPLICATED KEYS:", modeling_matrix_v1.duplicated(
    subset=["ID_PLANO", "treatment_line_primary", "UF_PRESTADOR", "MES"]
).sum())

SHAPE: (11649, 26)
CURRENT X SHAPE: (11526, 8)
CURRENT X COLUMNS: ['DEMAND_LAG_1M', 'DEMAND_LAG_3M', 'DEMAND_LAG_6M', 'DEMAND_MEAN_3M', 'DEMAND_MEAN_6M', 'DEMAND_STD_3M', 'PORTE_ORDINAL', 'MONTH']
ROWS EXCLUDED FOR UNKNOWN/PARTIAL TARGET: 123

COLUMNS:
['ID_PLANO', 'MES', 'UF_PRESTADOR', 'treatment_line_primary', 'STATUS_ATUACAO', 'volume_demanda_proxy', 'TARGET_MEASUREMENT_STATUS', 'TARGET_USABLE_FOR_TRAINING', 'IS_STRUCTURAL_ZERO', 'N_SOURCE_ROWS', 'N_BUNDLED_ROWS', 'BUNDLING_RATE', 'DEMAND_LAG_1M', 'DEMAND_LAG_3M', 'DEMAND_LAG_6M', 'DEMAND_MEAN_3M', 'DEMAND_MEAN_6M', 'DEMAND_STD_3M', 'DEMAND_GROWTH_MOM', 'PORTE', 'PORTE_ORDINAL', 'MONTH', 'QUARTER', 'n_municipality_pairs_total', 'n_municipality_pairs_active', 'share_active_municipality_pairs']

DUPLICATED KEYS: 0


In [15]:
print("NULLS:")
print(modeling_matrix_v1.isna().sum())
print("\nHEAD(10):")
modeling_matrix_v1.head(10)

NULLS:
ID_PLANO                              0
MES                                   0
UF_PRESTADOR                          0
treatment_line_primary                0
STATUS_ATUACAO                        0
volume_demanda_proxy                 12
TARGET_MEASUREMENT_STATUS             0
TARGET_USABLE_FOR_TRAINING            0
IS_STRUCTURAL_ZERO                    0
N_SOURCE_ROWS                         0
N_BUNDLED_ROWS                        0
BUNDLING_RATE                      5277
DEMAND_LAG_1M                       129
DEMAND_LAG_3M                      1504
DEMAND_LAG_6M                      4512
DEMAND_MEAN_3M                      272
DEMAND_MEAN_6M                      383
DEMAND_STD_3M                       836
DEMAND_GROWTH_MOM                  5018
PORTE                                 0
PORTE_ORDINAL                         0
MONTH                                 0
QUARTER                               0
n_municipality_pairs_total            0
n_municipality_pairs_active      

,ID_PLANO,MES,UF_PRESTADOR,treatment_line_primary,STATUS_ATUACAO,volume_demanda_proxy,TARGET_MEASUREMENT_STATUS,TARGET_USABLE_FOR_TRAINING,IS_STRUCTURAL_ZERO,N_SOURCE_ROWS,...,DEMAND_MEAN_6M,DEMAND_STD_3M,DEMAND_GROWTH_MOM,PORTE,PORTE_ORDINAL,MONTH,QUARTER,n_municipality_pairs_total,n_municipality_pairs_active,share_active_municipality_pairs
0,1019993,2025-09-01,TO,adjuvant,ATUANTE,0.0,STRUCTURAL_ZERO_NO_CLAIM,True,True,0.0,...,0.166667,0.000000,NaN,GRANDE,2,9,3,4.0,1.0,0.250000
1,1019993,2025-09-01,PA,advanced,ATUANTE,0.0,STRUCTURAL_ZERO_NO_CLAIM,True,True,0.0,...,0.166667,0.577350,NaN,GRANDE,2,9,3,4.0,1.0,0.250000
2,1019993,2025-09-01,TO,advanced,ATUANTE,0.0,STRUCTURAL_ZERO_NO_CLAIM,True,True,0.0,...,0.000000,0.000000,NaN,GRANDE,2,9,3,4.0,1.0,0.250000
3,1019993,2025-09-01,TO,neoadjuvant,ATUANTE,0.0,STRUCTURAL_ZERO_NO_CLAIM,True,True,0.0,...,0.666667,0.577350,0.0,GRANDE,2,9,3,4.0,1.0,0.250000
4,1169447,2025-11-01,PA,adjuvant,ATUANTE,0.0,STRUCTURAL_ZERO_NO_CLAIM,True,True,0.0,...,0.166667,0.577350,NaN,PEQUENO,0,11,4,3.0,1.0,0.333333
5,1169447,2025-12-01,PA,adjuvant,ATUANTE,2.0,OBSERVED_MEASURED,True,False,2.0,...,0.166667,0.577350,-1.0,PEQUENO,0,12,4,3.0,1.0,0.333333
6,1169447,2025-11-01,PA,neoadjuvant,ATUANTE,0.0,STRUCTURAL_ZERO_NO_CLAIM,True,True,0.0,...,1.166667,1.000000,1.0,PEQUENO,0,11,4,3.0,1.0,0.333333
7,1169447,2025-12-01,PA,neoadjuvant,ATUANTE,1.0,OBSERVED_MEASURED,True,False,1.0,...,1.166667,1.000000,-1.0,PEQUENO,0,12,4,3.0,1.0,0.333333
8,1169698,2025-04-01,AM,adjuvant,ATUANTE,0.0,STRUCTURAL_ZERO_NO_CLAIM,True,True,0.0,...,0.666667,1.154701,NaN,MÉDIO,1,4,2,3.0,1.0,0.333333
9,1169698,2025-05-01,AM,adjuvant,ATUANTE,3.0,OBSERVED_MEASURED,True,False,3.0,...,0.500000,1.154701,-1.0,MÉDIO,1,5,2,4.0,1.0,0.250000


## 8. Validation

### 8.1 Target

In [16]:
target_status_counts = modeling_matrix_v1["TARGET_MEASUREMENT_STATUS"].value_counts()
print("Target measurement status:")
print(target_status_counts)
print("\nBundling rate among observed affected rows:")
print(modeling_matrix_v1.loc[modeling_matrix_v1["BUNDLING_RATE"].gt(0), "BUNDLING_RATE"].describe())

all_bundled = modeling_matrix_v1["TARGET_MEASUREMENT_STATUS"].eq("OBSERVED_ALL_BUNDLED_UNKNOWN")
partially_bundled = modeling_matrix_v1["TARGET_MEASUREMENT_STATUS"].eq("OBSERVED_PARTIALLY_BUNDLED")
structural_zero = modeling_matrix_v1["TARGET_MEASUREMENT_STATUS"].eq("STRUCTURAL_ZERO_NO_CLAIM")

assert modeling_matrix_v1.loc[all_bundled, "volume_demanda_proxy"].isna().all()
assert (~modeling_matrix_v1.loc[all_bundled | partially_bundled, "TARGET_USABLE_FOR_TRAINING"]).all()
assert modeling_matrix_v1.loc[structural_zero, "volume_demanda_proxy"].eq(0).all()
assert modeling_matrix_v1.loc[structural_zero, "TARGET_USABLE_FOR_TRAINING"].all()

usable_target = modeling_matrix_v1.loc[training_mask, "volume_demanda_proxy"]
print("\nUsable target sum:", usable_target.sum())
print(usable_target.describe())
print("Structural zeros:", int(structural_zero.sum()))
print("Rows excluded for bundled target uncertainty:", int((all_bundled | partially_bundled).sum()))
print("Usable target skew:", usable_target.skew())

Target measurement status:
TARGET_MEASUREMENT_STATUS
OBSERVED_MEASURED               6249
STRUCTURAL_ZERO_NO_CLAIM        5277
OBSERVED_PARTIALLY_BUNDLED       111
OBSERVED_ALL_BUNDLED_UNKNOWN      12
Name: count, dtype: int64

Bundling rate among observed affected rows:
count    123.000000
mean       0.291048
std        0.270699
min        0.021739
25%        0.111111
50%        0.200000
75%        0.333333
max        1.000000
Name: BUNDLING_RATE, dtype: float64

Usable target sum: 597046.0
count    11526.000000
mean        51.799931
std        200.857833
min          0.000000
25%          0.000000
50%          1.000000
75%          7.000000
max       6974.000000
Name: volume_demanda_proxy, dtype: float64
Structural zeros: 5277
Rows excluded for bundled target uncertainty: 123
Usable target skew: 12.386069145631332


### 8.2 Keys

In [17]:
assert (modeling_matrix_v1["STATUS_ATUACAO"] == "ATUANTE").all(), "every row must be ATUANTE as of its own month"
# As-of eligibility re-checked independently of Section 6.1: every row comes strictly after
# its combination's first claim, and the operator had claims before the row's month.
first_claim = target.groupby(GROUP_KEYS_NO_MONTH, observed=True)["MES"].min().rename("_combo_first").reset_index()
chk = modeling_matrix_v1[GROUP_KEYS_NO_MONTH + ["MES"]].merge(first_claim, on=GROUP_KEYS_NO_MONTH, how="left")
assert (chk["MES"] > chk["_combo_first"]).all(), "a row precedes its combination's first claim"
print("As-of eligibility holds for every row.")
assert set(X_current.columns) <= approved_features, "X_current must contain APPROVED features only"
assert X_current.index.equals(modeling_matrix_v1.index[training_mask])
print("Duplicated keys:", modeling_matrix_v1.duplicated(
    subset=["ID_PLANO", "treatment_line_primary", "UF_PRESTADOR", "MES"]
).sum())
print("Missing in keys:", modeling_matrix_v1[["ID_PLANO", "treatment_line_primary", "UF_PRESTADOR", "MES"]].isna().sum().to_dict())
print("\nRows per operator:")
print(modeling_matrix_v1.groupby("ID_PLANO", observed=True).size().describe())
print("\nRows per month:")
print(modeling_matrix_v1.groupby("MES", observed=True).size())


As-of eligibility holds for every row.
Duplicated keys: 0
Missing in keys: {'ID_PLANO': 0, 'treatment_line_primary': 0, 'UF_PRESTADOR': 0, 'MES': 0}

Rows per operator:
count    507.000000
mean      22.976331
std       22.017049
min        1.000000
25%        9.000000
50%       17.000000
75%       30.000000
max      163.000000
dtype: float64

Rows per month:
MES
2025-02-01     567
2025-03-01     792
2025-04-01     927
2025-05-01     943
2025-06-01    1060
2025-07-01    1099
2025-08-01    1169
2025-09-01    1249
2025-10-01    1228
2025-11-01    1289
2025-12-01    1326
dtype: int64


### 8.3 Features

In [18]:
numeric_cols = ["DEMAND_LAG_1M", "DEMAND_MEAN_3M", "DEMAND_STD_3M", "PORTE_ORDINAL", "BUNDLING_RATE",
                "n_municipality_pairs_total", "n_municipality_pairs_active", "share_active_municipality_pairs"]
for col in numeric_cols:
    s = modeling_matrix_v1[col]
    print(f"{col}: missing={s.isna().mean()*100:.1f}%  skew={s.skew():.2f}  var={s.var():.2f}  nunique={s.nunique()}")

print("\nSTATUS_ATUACAO x PORTE crosstab (redundancy check):")
op_level = modeling_matrix_v1.drop_duplicates("ID_PLANO")
print(pd.crosstab(op_level["STATUS_ATUACAO"], op_level["PORTE"]))

DEMAND_LAG_1M: missing=1.1%  skew=11.68  var=44427.51  nunique=462
DEMAND_MEAN_3M: missing=2.3%  skew=9.64  var=41287.56  nunique=944
DEMAND_STD_3M: missing=7.2%  skew=16.08  var=9699.95  nunique=1464
PORTE_ORDINAL: missing=0.0%  skew=-2.38  var=0.56  nunique=4
BUNDLING_RATE: missing=45.3%  skew=13.81  var=0.00  nunique=42
n_municipality_pairs_total: missing=0.0%  skew=3.13  var=346.91  nunique=99
n_municipality_pairs_active: missing=0.0%  skew=2.88  var=9.96  nunique=24
share_active_municipality_pairs: missing=0.0%  skew=1.30  var=0.08  nunique=183

STATUS_ATUACAO x PORTE crosstab (redundancy check):
PORTE           GRANDE  MÉDIO  PEQUENO  SEM BENEFICIÁRIOS
STATUS_ATUACAO                                           
ATUANTE            391     43       38                 35


### 8.4 Leakage audit -- MEDIUM/HIGH risk features only

In [19]:
leakage_audit = feature_proposal_matrix[
    feature_proposal_matrix["LEAKAGE_RISK"].str.contains("MEDIUM|HIGH", na=False)
][["FEATURE_NAME", "LEAKAGE_RISK", "TEMPORAL_AVAILABILITY", "STATUS"]]

leakage_audit

,FEATURE_NAME,LEAKAGE_RISK,TEMPORAL_AVAILABILITY,STATUS
4,PCT_FEMININO,HIGH,contemporaneous to target,CANDIDATE (only as a trailing/lagged version)
5,PCT_AGE_60PLUS,HIGH,contemporaneous,CANDIDATE (only as a trailing/lagged version)
7,N_MUNICIPIOS_BENEFICIARIO_TRAILING,MEDIUM (if trailing) / HIGH (if reusing nb08/1...,strictly historical window ending before predi...,"CANDIDATE -- requires new engineering, not dir..."
8,N_MUNICIPIOS_PRESTADOR_TRAILING,MEDIUM (if trailing) / HIGH (as-is),strictly historical window ending before predi...,CANDIDATE -- requires historical-window engine...
9,N_PRESTADORES_NO_GRUPO,HIGH until dated,unknown source reference/publication date; mus...,PENDING -- excluded from X until linkage/date/...
10,N_PRESTADORES,HIGH until dated,unknown source reference/publication date; mus...,PENDING -- excluded from X until linkage/date/...
12,MARKET_SHARE_MAX,HIGH until dated/basis known,unknown; source must be dated and available be...,PENDING -- excluded from X until linkage/date/...
13,MARKET_SHARE_TOP3,HIGH until dated/basis known,unknown; source must be dated and available be...,PENDING -- excluded from X until linkage/date/...
14,MARKET_SHARE_TOP5,HIGH until dated/basis known,unknown; source must be dated and available be...,PENDING -- excluded from X until linkage/date/...
15,HHI_NETWORK,HIGH until dated/basis known,unknown; source must be dated and available be...,PENDING -- excluded from X until linkage/date/...


## 9. Data Lineage

```
df_norte
   |
   +--> treatment_line_primary  [priority convention, validated nb 19 -- 99.67% multi-label overlap]
   +--> target = groupby(...).sum(QT_ITEM_EVENTO_INFORMADO)          (10,302 observed rows)
   |        --> classify measured, partially bundled and all-bundled observations
   |        --> preserve all-bundled quantities as unknown; partial sums are diagnostic only
   |
   +--> notebooks/11_activity_matrix_operator_municipality.ipynb
   |        --> activity_matrix_by_municipality.parquet (5,089 operator x municipality pairs)
   +--> notebooks/12_kmeans_operator_municipality.ipynb
   |        --> active_operator_municipality_pairs.parquet (874 active pairs, k=2)
   |
   +--> rollup(ID_PLANO) <- groupby+count of the two files above
   |        --> full-year STATUS_ATUACAO  [traceability + Section 4.2 only; selects no rows]
   |
   +--> activity_status(claims before s), for every month s   (Section 4.0)
   |        --> as-of STATUS_ATUACAO, n_municipality_pairs_* per operator-month
   |        (verified: on all claims it reproduces notebook 12 operator by operator)
   |
   +--> reindex every combination, operator's first claim -> end of data -> complete panel
   |        --> months without a claim become zero billed volume
   +--> + DEMAND_LAG/MEAN/STD (groupby.shift/rolling, strictly past-only)
   +--> keep rows ATUANTE as of their month AND whose combination was billed before it
   |        (Section 6.1: any split on MES is free of look-ahead)
   +--> + PORTE, PORTE_ORDINAL  [static merge, validated against duplication]
   +--> + MONTH, QUARTER
   v
modeling_matrix_v1 (in-memory analytical table; target diagnostics retained)
   |
   +--> retain rows with a usable target and select CURRENT_X_COLUMNS
   |        (STATUS_ATUACAO and annual activity fields are not predictors)
   v
X_current (approved predictors only)

Non-Hospital Market Share --> notebook 14 audit --> proposed network/concentration fields
   --> PENDING: identifier crosswalk + dated pre-cutoff availability + compatible
               geographic scope + aggregation rule + join cardinality
   --> not merged into modeling_matrix_v1 and not added to X_current
```

## 10. Recommendation

The modeling population is restricted, row by row, to operators that were `ATUANTE` as of
each row's month, on combinations already billed before it. `STATUS_ATUACAO` performs that
filter and is not a predictor in `X_current`.

**Current predictor contract**: `DEMAND_LAG_1M/3M/6M`, `DEMAND_MEAN_3M/6M`,
`DEMAND_STD_3M`, `PORTE_ORDINAL` and `MONTH`. These are the fields explicitly approved
and materialized in `X_current`. `SEASONALITY`, `QUARTER`, `PORTE` and
`NM_MODALIDADE_*` remain candidates or alternatives and are not presented as implemented
predictors.

**Unresolved integration requirements**:

- **`treatment_line_primary`**: the single biggest open item. The priority rule used
  throughout this notebook (`advanced > adjuvant > neoadjuvant > staging`) is an
  unvalidated methodological convention -- it affects 99.67% of events and must be
  validated before treatment-line results are interpreted.
- **`STATUS_ATUACAO` source of truth**: two segmentations exist in this repository (notebook 10,
  operator-level; notebooks 11/12, operator x municipality, used here) and they disagree
  for a meaningful share of operators (Section 4.2). The status is now computed as of each
  month, which removes the population-selection leakage of the earlier full-year version;
  which segmentation should define ATUANTE remains a definitional choice.
- **Target measurement completeness**: all-bundled groups remain unknown and partially
  bundled sums remain diagnostic lower bounds. Both are excluded from the target and from
  historical feature windows. Zeros are created from each operator's first claim to the end
  of the data; they are zero billed volume, a claims-based proxy rather than enrollment proof.
- **Network/concentration integration**: notebook 14 supports plan-level provider counts,
  maximum/top-three/top-five normalized shares and HHI as candidates. `N_PRESTADORES` is
  computed from `nunique(CD_CNES)` in the market-share source, never from `df_norte`. The
  declared `N_PRESTADORES_NO_GRUPO` remains plan-level, and the count-incomplete plan is
  retained as a diagnostic but excluded from feature roll-ups. All candidates remain outside
  `X_current` until identifier linkage, reference/publication dates, geographic scope,
  aggregation semantics and join cardinality pass. The national 27-state snapshot cannot be
  silently treated as North-only network evidence.
- **`MARKET_SHARE_TOTAL`**: retained only as a weight-conservation diagnostic because the
  normalized total is 1; it is not predictive.
- **`VERTICALIZATION`**: notebook 14 found no dated ownership evidence. Every ownership,
  intensity and verticalized-share field remains pending evidence and excluded from `X`;
  `VERTICALIZATION_TARGET = Unknown` is preserved and is never converted to zero.
- **`BENEFICIARY DATA`**: the ANS `DADOS_DE_PLANOS` beneficiary-count base was never
  ingested -- only `PORTE` (a coarse tier, not a real count) is available.
- **Historical breadth features** (`N_MUNICIPIOS_*_TRAILING`,
  `N_PROCEDIMENTOS_DISTINTOS_TRAILING`,
  `PCT_FEMININO`/`PCT_AGE_60PLUS` lagged): technically feasible but not built here --
  every window must end before the prediction month; full-year versions would leak future information.

No dataset was modified -- everything above reads from `data/processed/` and
`data/interim/` and writes only the versioned modeling output declared in Section 11.
The persisted output contains only keys, the usable target and the approved predictor contract.


## 10. Output

Only the training-safe view is persisted: model keys, the usable demand target and the
approved columns in `CURRENT_X_COLUMNS`. Full-year activity fields, population-filter fields,
target-quality diagnostics and pending network/verticalization candidates remain outside the
training artifact. A companion CSV records the exact feature contract.

Every persisted row satisfies the as-of eligibility of Section 6.1, so any consumer can split
the file on `MES` without look-ahead. The file name and its 13 columns are the stable contract consumed by notebooks 15,
16 and 17, so those notebooks need no path or schema changes.


In [20]:
OUTPUT_PATH = "../data/processed/north_region_mama_2025_modeling_matrix_atuantes_v1.parquet"
CONTRACT_PATH = "../data/processed/north_region_mama_2025_modeling_matrix_atuantes_v1_feature_contract.csv"
SAFE_TRAINING_COLUMNS = UNIT_KEYS + ["volume_demanda_proxy"] + CURRENT_X_COLUMNS
training_matrix_v1 = modeling_matrix_v1.loc[training_mask, SAFE_TRAINING_COLUMNS].copy()

assert len(training_matrix_v1) == len(X_current) == len(y_current)
assert training_matrix_v1[UNIT_KEYS].duplicated().sum() == 0
assert training_matrix_v1["volume_demanda_proxy"].notna().all()
assert set(training_matrix_v1.columns) == set(SAFE_TRAINING_COLUMNS)
assert "STATUS_ATUACAO" not in training_matrix_v1.columns

contract_fields = ["FEATURE_NAME", "SOURCE_DATASET", "SOURCE_COLUMN", "TRANSFORMATION",
                   "AGGREGATION", "TEMPORAL_AVAILABILITY", "STATUS"]
predictor_contract = feature_proposal_matrix.set_index("FEATURE_NAME").loc[
    CURRENT_X_COLUMNS, contract_fields[1:]
].reset_index()
predictor_contract.insert(1, "ROLE", "PREDICTOR")
key_target_contract = pd.DataFrame([
    dict(FEATURE_NAME="ID_PLANO", ROLE="KEY", SOURCE_DATASET="df_norte", SOURCE_COLUMN="ID_PLANO",
         TRANSFORMATION="none", AGGREGATION="grouping key", TEMPORAL_AVAILABILITY="available on each claim",
         STATUS="KEY -- not a predictor; row present only if ATUANTE as of MES (prior claims) and the combination was billed before MES"),
    dict(FEATURE_NAME="treatment_line_primary", ROLE="KEY", SOURCE_DATASET="df_norte", SOURCE_COLUMN="treatment-line flags",
         TRANSFORMATION="priority convention; validation pending", AGGREGATION="grouping key", TEMPORAL_AVAILABILITY="available on each claim", STATUS="KEY -- interpretation pending"),
    dict(FEATURE_NAME="UF_PRESTADOR", ROLE="KEY", SOURCE_DATASET="df_norte", SOURCE_COLUMN="UF_PRESTADOR",
         TRANSFORMATION="none", AGGREGATION="grouping key", TEMPORAL_AVAILABILITY="available on each claim", STATUS="KEY -- not a predictor"),
    dict(FEATURE_NAME="MES", ROLE="KEY", SOURCE_DATASET="df_norte", SOURCE_COLUMN="DT_REALIZACAO",
         TRANSFORMATION="calendar month", AGGREGATION="grouping key", TEMPORAL_AVAILABILITY="target month", STATUS="KEY -- not a predictor"),
    dict(FEATURE_NAME="volume_demanda_proxy", ROLE="TARGET", SOURCE_DATASET="df_norte", SOURCE_COLUMN="QT_ITEM_EVENTO_INFORMADO",
         TRANSFORMATION="observed sum; zero for months without a claim, from the operator's first claim to the end of the data",
         AGGREGATION="sum by model keys; bundled-uncertain rows excluded", TEMPORAL_AVAILABILITY="forecast outcome", STATUS="TARGET"),
])
feature_contract = pd.concat([key_target_contract, predictor_contract], ignore_index=True)
assert feature_contract["FEATURE_NAME"].tolist() == SAFE_TRAINING_COLUMNS

training_matrix_v1.to_parquet(OUTPUT_PATH, index=False)
feature_contract.to_csv(CONTRACT_PATH, index=False)
print(f"Saved training matrix: {OUTPUT_PATH}")
print(f"Saved feature contract: {CONTRACT_PATH}")
print(f"Training shape: {training_matrix_v1.shape}")
print(f"Distinct operators: {training_matrix_v1['ID_PLANO'].nunique()}")
feature_contract


Saved training matrix: ../data/processed/north_region_mama_2025_modeling_matrix_atuantes_v1.parquet
Saved feature contract: ../data/processed/north_region_mama_2025_modeling_matrix_atuantes_v1_feature_contract.csv
Training shape: (11526, 13)
Distinct operators: 507


,FEATURE_NAME,ROLE,SOURCE_DATASET,SOURCE_COLUMN,TRANSFORMATION,AGGREGATION,TEMPORAL_AVAILABILITY,STATUS
0,ID_PLANO,KEY,df_norte,ID_PLANO,none,grouping key,available on each claim,KEY -- not a predictor; row present only if AT...
1,treatment_line_primary,KEY,df_norte,treatment-line flags,priority convention; validation pending,grouping key,available on each claim,KEY -- interpretation pending
2,UF_PRESTADOR,KEY,df_norte,UF_PRESTADOR,none,grouping key,available on each claim,KEY -- not a predictor
3,MES,KEY,df_norte,DT_REALIZACAO,calendar month,grouping key,target month,KEY -- not a predictor
4,volume_demanda_proxy,TARGET,df_norte,QT_ITEM_EVENTO_INFORMADO,"observed sum; zero for months without a claim,...",sum by model keys; bundled-uncertain rows excl...,forecast outcome,TARGET
5,DEMAND_LAG_1M,PREDICTOR,modeling_matrix_v1,volume_demanda_proxy,shift(1) within group,none,strictly prior to target,APPROVED
6,DEMAND_LAG_3M,PREDICTOR,modeling_matrix_v1,volume_demanda_proxy,shift(3),none,prior,APPROVED
7,DEMAND_LAG_6M,PREDICTOR,modeling_matrix_v1,volume_demanda_proxy,shift(6),none,prior,APPROVED
8,DEMAND_MEAN_3M,PREDICTOR,modeling_matrix_v1,volume_demanda_proxy,shift(1).rolling(3).mean(),mean,prior (entire window in the past),APPROVED
9,DEMAND_MEAN_6M,PREDICTOR,modeling_matrix_v1,volume_demanda_proxy,shift(1).rolling(6).mean(),mean,prior,APPROVED


## References

1. Agência Nacional de Saúde Suplementar (ANS). Dados abertos: TISS Ambulatorial [Internet]. Brasília: ANS; 2026 [cited 2026 Sep 11]. Available from: https://dadosabertos.ans.gov.br/FTP/PDA/TISS/AMBULATORIAL/
2. Sociedade Brasileira de Oncologia Clínica (SBOC). Diretrizes SBOC 2025: câncer de mama — estadiamento, neoadjuvante, adjuvante e doença avançada. São Paulo: SBOC; 2025.
3. Kaufman S, Rosset S, Perlich C, Stitelman O. Leakage in data mining: formulation, detection, and avoidance. ACM Trans Knowl Discov Data. 2012;6(4):15.
4. James G, Witten D, Hastie T, Tibshirani R. An introduction to statistical learning. New York: Springer; 2013.
5. Pedregosa F, Varoquaux G, Gramfort A, Michel V, Thirion B, Grisel O, et al. Scikit-learn: machine learning in Python. J Mach Learn Res. 2011;12:2825-30.
